## **Tutorial: Accessing Precipitation Timeseries Data**
In this short tutorial, you will learn how to:
* Efficiently retrieve and display precipitation time series data
* Visualize the precipitation data
* Export retrieved precipitation data to a CSV or Parquet file

This notebook queries the Multi-Radar/Multi-Sensor (MRMS) precipitation data stored in an [AWS cloud bucket](https://noaa-mrms-parquet-pds.s3.us-east-1.amazonaws.com/index.html) to generate a continuous time series from the data. This data may also be accessed using the interactive National Oceanic and Atmospheric Administration [U.S. Precipitation Time Series Explorer (PTE)](http://ncei.noaa.gov/products/precipitation-time-series-explorer/), where precipitation data can be filtered using a geospatial application.

This notebook contains multiple sections that guide you through accessing the data from the bucket, processing the data, and creating visualizations. The Table of Contents contains detailed headers to direct you to the different parts of the notebook. The cells are also numbered to match the sections in the Table of Contents to make it easier to navigate through the notebook. **It is recommended that you read through the entire notebook before running it.** If you want to make changes to the notebook, make sure to save a copy. 

In order for this notebook to work correctly, Sections 1 and 2 have to be run before the rest of the sections in the notebook. These sections contain the import of the Python modules that are needed for the data analysis and plotting, and the custom functions that are used to retrieve and display the precipitation time series data. Section 3 guides you through the process of defining the data set that you would like to retrieve from the AWS cloud bucket, Section 4 contains a workflow to extract and display a time series for a single location, and Section 5 walks through the process of extracting and displaying time series for multiple locations. The information that you enter in Section 3 (dataset, state, and year) are used in Sections 4 and 5. If you would like to change the dataset, state, or year that you are examining, then you will need to change your inputs in Section 3, rerun the cell, and continue through Sections 4 and 5. Examples for these query parameters are included so the notebook runs from start to finish. You can adjust the query parameters to customize the notebook for your use. This notebook includes custom warnings and error messages designed to guide you in providing the correct information needed to successfully query the AWS cloud bucket and generate the figures. You are able to download the time series for a single location (Section 4) or multiple locations (Section 5) as a CSV or Parquet file. Sections 4 and 5 contain alt-text for the figures that will display to the screen. This alt-text will adjust if you change the inputs to the figures. You are able to save the figures, but the alt-text is not embedded.

The purpose of this notebook is to provide examples of how you can interact with the AWS bucket to retrieve data, process the data to obtain a time series, and various ways to visualize the data. This notebook was designed to accommodate the most typical use cases, so some features may not work as well if you are attempting to visualize a long time series or many locations. Feel free to modify the notebook to fit your needs. If you would like to modify the notebook, you will need to save a copy to your Google Drive or local machine. This notebook has been verified to work with the MultiSensor_QPE_01H_Pass2_00.00 and FLASH_QPE_FFGMAX_00.00 data sets. This notebook may or may not work with other data sets that are present in the MRMS AWS Cloud bucket.

This notebook was created as part of an initiative to develop industry-tailored resources. For more information, visit the [Our Impact](https://www.ncei.noaa.gov/about/our-impact) website. For questions about this notebook, contact industryproving.grounds@noaa.gov.

---
## **1.0 Import Modules**
**The Goal:**
To initialize the Python environment by importing the necessary libraries for data access (`s3fs`), processing (`polars`), and visualization (`matplotlib`).

**How to use this section:**
1.  **Run the cell below:** This will check your environment.
2.  **Wait for the check:**
    * If you are in **Google Colab**, it will automatically install missing packages.
    * If you are on a **Local Machine**, it will check for dependencies and warn you if anything is missing.

**Key Libraries Used:**
* **`s3fs`**: This is our bridge to the cloud. It allows Python to treat an Amazon S3 bucket like a local file system, enabling us to list folders and open files remotely.
* **`polars`**: A high-performance dataframe library (similar to pandas but faster). We use it to handle the large precipitation datasets efficiently.
* **`matplotlib`**: The standard library for creating static, animated, and interactive visualizations in Python.

In [ ]:
# --- 1.0: Import Modules & Setup Environment ---

import sys
import subprocess
import importlib

def check_and_install_packages():
    """
    Checks for required third-party packages and installs them if missing.
    This works for both Google Colab and local environments.
    """
    # Mapping of import_name -> pip_package_name (usually the same, but good to be explicit)
    required_packages = {
        "s3fs": "s3fs",
        "pyarrow": "pyarrow",
        "polars": "polars",
        "matplotlib": "matplotlib"
    }

    missing_packages = []

    print("--- Checking System Dependencies ---")
    for import_name, package_name in required_packages.items():
        try:
            importlib.import_module(import_name)
            # print(f"✅ {package_name} is already installed.")
        except ImportError:
            print(f"⚠️ {package_name} is missing.")
            missing_packages.append(package_name)

    if missing_packages:
        print(f"\n📦 Installing missing packages: {', '.join(missing_packages)}...")
        try:
            # Construct pip install command
            cmd = [sys.executable, "-m", "pip", "install", "-q"] + missing_packages

            # Run installation
            subprocess.check_call(cmd)
            print("✅ Installation complete.\n")

            # Note: In some kernels (like Jupyter), a restart might be required
            # to load new packages, but often importlib handles it fine for simple scripts.
        except subprocess.CalledProcessError as e:
            print("❌ Error during installation.")
            print(f"   Command failed: {e}")
            print("   Please try running the following command manually in a new cell:")
            print(f"   !pip install {' '.join(missing_packages)}")
            # Don't exit, try to proceed in case it was a partial success
    else:
        print("✅ All dependencies are present.\n")

# Run the dependency check
check_and_install_packages()


# --- Standard Library Imports ---
import os
import re
import zipfile
import traceback
import textwrap
import warnings
import numpy as np
from datetime import datetime, timedelta, timezone
from math import radians, sin, cos, sqrt, atan2
from io import BytesIO

# --- Third-Party Imports ---
print("--- Importing Modules ---")
try:
    # Data Handling
    import pyarrow.dataset as ds
    import pyarrow.parquet as pq
    import polars as pl

    # S3 & File System
    import s3fs

    # Plotting
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    import matplotlib.ticker as ticker

    print("✅ Libraries imported successfully.")

except ImportError as e:
    print(f"\n❌ Import Error: {e}")
    print("   If you just installed packages, you may need to restart the runtime/kernel.")

---
## **2.0 Custom Helper Functions**
This section defines custom utility functions (or "helpers") built specifically for this notebook. These functions handle all the complex or repetitive tasks, allowing the main workflow steps (Sections 3.0, 4.0, and 5.0) to focus solely on configuration and execution.

**The Goal:**
The primary goal of this section is to load all the necessary tools into memory. You do **not** need to edit any code here, but understanding the purpose of each group is helpful as they form the foundation for the main data extraction and visualization workflows.

**How to use this section:**
1.  **Run the cells below:** You do not need to edit any code here.
2.  **Verify:** Ensure the cell runs without error. Once run, these functions are available for the rest of the session.

**Overview of key Helper Functions:**
* **S3 Helpers:** Handle the connection to NOAA's S3 data repository and list available data files.
* **Validation Helpers:** Ensure coordinates and dates are valid before running heavy calculations.
* **Geospatial Helpers:** Since climate data comes on a fixed grid, these functions calculate the **"Nearest Neighbor"**—finding the grid point closest to your specific latitude/longitude.
* **Plotting Helpers:** Handle the complex logic of formatting time-series charts and creating accessibility-friendly text descriptions (Alt Text).
* **Data Export Helpers:** Manage creating unique, descriptive filenames and saving your final data (CSV/Parquet) and plots (PNG) to your local machine.
* **Environment Helper:** Performs a check to determine if the notebook is running in a cloud environment (like Google Colab) to correctly enable or disable automatic file downloads.

### **2.1 S3 & Data Access Helpers**
**The Goal:**
These functions are your gateway to the cloud. They handle the low-level tasks of establishing a connection to the remote S3 data repository (using `s3fs`), navigating the file structure to locate the data, and securely pulling raw Parquet data into a Polars DataFrame for processing.

**How to use this cell:**
* **Run the cell.** (No user input required).

In [ ]:
# --- 2.1: S3 & Data Access Helpers ---

def log_user_error(message, hint=None, details=None):
    """
    Centralized error logging for user-facing issues.
    Separates friendly advice from technical debug info.

    Args:
        message (str): The main friendly error message.
        hint (str, optional): Actionable advice for the user.
        details (str, optional): Technical info (paths, exceptions) for debugging.
    """
    print(f"\n❌ ACTION REQUIRED: {message}")
    if hint:
        print(f"   Hint: {hint}")

    if details:
        print("-" * 60)
        print(f"   Debug Details: {details}")
    print("-" * 60 + "\n")


def get_s3_filesystem(anon=True):
    """
    Initializes and returns an S3FileSystem object.

    Caches the filesystem object in the global scope as 's3'
    to avoid repeated initializations.

    Args:
        anon (bool): Whether to initialize in anonymous mode.

    Returns:
        s3fs.S3FileSystem: An initialized S3 filesystem object.
    """
    # Check if 's3' is already in the global scope and is valid
    if 's3' in globals() and isinstance(globals()['s3'], s3fs.S3FileSystem):
        return globals()['s3']

    print("Initializing new S3 connection...")
    try:
        s3 = s3fs.S3FileSystem(anon=anon)
        # Store it in globals for reuse
        globals()['s3'] = s3
        return s3
    except Exception as e:
        log_user_error(
            "Failed to initialize S3 filesystem.",
            hint="Check your internet connection.",
            details=f"Exception: {e}"
        )
        traceback.print_exc()
        return None


def examine_s3_path(s3_path, filesystem, num_columns=4):
    """
    Lists the contents of a given S3 path in a multi-column format.

    Args:
        s3_path (str): The S3 path to a directory (e.g., 's3://bucket/folder/').
        filesystem (s3fs.S3FileSystem): An initialized S3 filesystem object.
        num_columns (int): The number of columns to use for the output display.

    Returns:
        list: A list of dictionaries with details about each item, or an empty list.
    """
    print(f"--- Examining S3 Path: {s3_path} ---")
    try:
        contents = filesystem.ls(s3_path, detail=True)
        if not contents:
            print("Path not found or is empty.")
            return []

        directories = [item['name'] for item in contents if item['type'] == 'directory']
        files = [item['name'] for item in contents if item['type'] == 'file']

        print(f"\nFound {len(directories)} director(y/ies) and {len(files)} file(s).")

        # --- FORMATTING LOGIC ---
        def print_in_columns(item_list, title):
            if not item_list:
                return
            print(f"\n{title}:")
            # Clean up names and sort them
            names = sorted([os.path.basename(p).replace('state=', '').replace(
                'year=', '') for p in item_list])
            # Calculate column width based on the longest item name
            col_width = max(len(name) for name in names) + 4

            # Print items in a grid
            for i in range(0, len(names), num_columns):
                # Get a "row" of items
                row = names[i:i + num_columns]
                # Print each item in the row with padding
                print("".join(f"{name:<{col_width}}" for name in row))

        print_in_columns(directories, "Directories")
        print_in_columns(files, "Files")

        print("\n--- Examination Complete ---")
        return contents

    except Exception as e:
        log_user_error(
            "Failed to examine the S3 path.",
            details=f"Path: {s3_path}\nException: {e}"
        )
        # traceback.print_exc() # Optional: hide traceback if log_user_error captures enough
        return []


def list_s3_contents(base_s3_path, filesystem, content_type='all'):
    """
    Lists directories or files within a specified S3 path.
    (REVISED: Uses os.path.basename for robust relative path extraction)

    Args:
        base_s3_path (str): The S3 path to list (e.g., 's3://bucket/folder/').
        filesystem (s3fs.S3FileSystem): An initialized S3 filesystem object.
        content_type (str): Type of content to list: 'all', 'directories', or 'files'.

    Returns:
        list: A list of the final component names (directory or file names),
              or an empty list if the path doesn't exist or an error occurs.
    """
    # Ensure the path ends with a '/' for directory listing.
    # Logic: If path contains '/' after the scheme (s3://), it's a sub-path (like a dataset).
    # We append '/' to force s3fs to list *contents* rather than the directory itself.
    # This fixes discovery for root-level datasets.
    path_no_scheme = base_s3_path.split('://')[1]
    if not base_s3_path.endswith('/') and '/' in path_no_scheme:
         list_path = base_s3_path + '/'
    else:
         list_path = base_s3_path

    try:
        # Get detailed listing
        contents = filesystem.ls(list_path, detail=True)

        results = []
        for item in contents:
            item_type = item['type']
            item_name = item['name'] # Full path, e.g., bucket/folder/subfolder

            # Remove trailing slash if it's a directory before getting basename
            cleaned_path = item_name.rstrip('/')
            relative_name = os.path.basename(cleaned_path)

            if content_type == 'all':
                results.append(relative_name)
            elif content_type == 'directories' and item_type == 'directory':
                results.append(relative_name) # Already stripped trailing slash
            elif content_type == 'files' and item_type == 'file':
                results.append(relative_name)

        return results

    except FileNotFoundError:
        # This acts as an internal check in some flows, so we can stick to print
        print(f"Info: Path not found or is empty: {list_path}")
        return []
    except Exception as e:
        log_user_error(
            f"Error listing S3 contents.",
            details=f"Path: {list_path}\nException: {e}"
        )
        traceback.print_exc()
        return []


def validate_s3_data_path(base_s3_path, state_to_check, filesystem, num_columns=5):
    """
    Performs a robust, multi-level check to validate a data path on S3.

    This function verifies a state directory exists, then checks EACH year
    subdirectory to confirm it contains data files. It returns only the list
    of years that are confirmed to have data.

    Args:
        base_s3_path (str): The top-level S3 path containing state directories.
        state_to_check (str): The name of the state to validate (e.g., "Delaware").
        filesystem (s3fs.S3FileSystem): An initialized S3 filesystem object.
        num_columns (int): The number of columns for printing the year list.

    Returns:
        list: A sorted list of "valid" year strings (those containing data),
              otherwise None.
    """
    print(f"\n--- Validating data path for state: {state_to_check} ---")
    try:
        #Check if the user's chosen state directory exists.
        state_path = f"{base_s3_path.rstrip('/')}/state={state_to_check}"
        if not filesystem.exists(state_path):
            log_user_error(
                "State directory not found.",
                hint=f"Check the spelling of '{state_to_check}' or confirm the dataset " \
                        f"covers this region.",
                details=f"Searched at: {state_path}"
            )
            return None
        print(f"✅ State directory found.")

        #Find all 'year' subdirectories.
        year_items = filesystem.ls(state_path, detail=True)
        year_dirs = [item['name'] for item in year_items if item['type'] == 'directory']
        if not year_dirs:
            log_user_error(
                "No 'year' subdirectories found for this state.",
                details=f"Searched at: {state_path}"
            )
            return None

        print("Verifying data availability in each year directory...")
        valid_years = []
        empty_years = []

        #Loop through each year and check for parquet files.
        for year_path in year_dirs:
            year_name = os.path.basename(year_path).replace('year=', '')
            parquet_files = filesystem.glob(f"{year_path}/*.parquet")
            if parquet_files:
                valid_years.append(year_name)
            else:
                empty_years.append(year_name)

        #Check if ANY valid years were found.
        if not valid_years:
            log_user_error(
                "No .parquet files found in any of the year directories.",
                hint="The directories exist but appear empty.",
                details=f"State Path: {state_path}"
            )
            return None

        #Report the findings to the user.
        valid_years.sort()
        print(f"\n✅ Found {len(valid_years)} year(s) with data available:")

        # Print the list of valid years in a clean, multi-column format
        col_width = max(len(year) for year in valid_years) + 4
        for i in range(0, len(valid_years), num_columns):
            row = valid_years[i:i + num_columns]
            print("".join(f"{name:<{col_width}}" for name in row))

        # Warn the user about any empty year directories that were found
        if empty_years:
            print(f"\n⚠️ Warning: The following year directories were found but are empty: " \
                  f"{', '.join(sorted(empty_years))}")

        print("\n--- Path Validation Successful ---")
        return valid_years

    except Exception as e:
        log_user_error(
            "Path Validation Failed.",
            details=f"{e}"
        )
        return None

def retrieve_data_from_s3(s3_path, filesystem):
    """
    Retrieves and scans MRMS parquet data from a given S3 path lazily.

    Args:
        s3_path (str): The S3 path to the data directory (e.g., 's3://...').
        filesystem (s3fs.S3FileSystem): An initialized S3 filesystem object.

    Returns:
        pl.LazyFrame: A Polars LazyFrame with the scanned data query plan, or an empty
            LazyFrame if no data is found or an error occurs.
    """
    print(f"Searching for data at: {s3_path}...")
    try:
        # S3 storage options for native, anonymous Polars/Rust parquet scanning
        storage_options = {"skip_signature": "true"}

        # Ensure path ends with wildcard to scan all Parquet files in the subdirectory
        s3_uri = f"{s3_path}/*.parquet"

        # Create a lazy query plan to scan the Parquet files directly from S3
        precip_pl_df = pl.scan_parquet(
            s3_uri,
            storage_options=storage_options,
            hive_partitioning=True
        )
        print("Successfully created lazy query plan for S3 Parquet files.")
        return precip_pl_df

    except Exception as e:
        log_user_error(
            "Failed to retrieve or process data from S3.",
            details=f"{e}"
        )
        traceback.print_exc()
        return pl.LazyFrame()

def discover_s3_path_iteratively(base_s3_path, partition_choices, s3_filesystem):
    """
    Performs an iterative discovery of an S3 path based on user choices.

    It allows a user to "click-down" through an unknown S3
    partition scheme.

    Args:
        base_s3_path (str): The starting S3 path (e.g., "s3://.../DATASET").
        partition_choices (list): A list of user-selected sub-partitions
                                  (e.g., ['parquet', 'state=Washington']).
        s3_filesystem (s3fs.S3FileSystem): An initialized S3 filesystem object.

    Returns:
        tuple: A (str, int) tuple of (validated_path, selected_year)
               Returns (None, None) if validation is not complete or fails.
    """
    try:
        #Build the path to examine
        current_path_to_list = base_s3_path
        if partition_choices:
            current_path_to_list = f"{base_s3_path}/{'/'.join(partition_choices)}"

        print(f"--- Examining S3 Path: {current_path_to_list} ---")

        #Discover all sub-directories at the current path
        available_dirs = list_s3_contents(
            current_path_to_list, s3_filesystem, content_type='directories'
        )

        #Check for .parquet files at the *current* level
        data_files = s3_filesystem.glob(f"{current_path_to_list}/*.parquet")

        # --- DECISION & ACTION ---

        if not available_dirs and not data_files:
            log_user_error(
                "No subdirectories or .parquet files found at this path.",
                hint="Please check your dataset name and choices in the 'partition_choices' list.",
                details=f"Path: {current_path_to_list}"
            )
            return (None, None)

        elif data_files:
            # --- VALIDATION SUCCESS ---
            print(f"✅ VALIDATION SUCCESS: Data files found at: {current_path_to_list}")
            print("   This is the final data level.")

            validated_path = current_path_to_list
            selected_year = None

            # --- Parse and set the year variable ---
            for part in partition_choices:
                if part.strip().startswith("year="):
                    try:
                        year_val = int(part.split("=")[1])
                        selected_year = year_val
                        print(f"   Set 'selected_year' to: {selected_year}")
                        break
                    except Exception:
                        print(f"   (Could not parse year from '{part}')")

            if selected_year is None:
                print("   (Info: No 'year=...' partition was found. Year validation will be skipped.)")

            print("\n👉 ACTION REQUIRED:")
            print("   You can now proceed to Section 4.0 (Single Location) or Section 5.0 (Multi-Location)")
            print("   to extract, visualize, and save specific time series data.")

            # Return the validated path and year
            return (validated_path, selected_year)

        else:
            # --- MORE DIRECTORIES FOUND ---
            print(f"\n✅ Found {len(available_dirs)} available sub-partitions:")
            for partition in sorted(available_dirs):
                print(f"   - {partition}")

            print("\n👉 ACTION REQUIRED:")
            print("   1. Choose ONE partition from the list above.")
            print(f"   2. Add your choice (as a string) to the 'partition_choices' list")
            print("      at the top of the previous cell.")
            print("   3. Re-run the previous cell.")

            print(f"\n   Current Path Components: {partition_choices}")
            return (None, None)

    except Exception as e:
        log_user_error(
            "An error occurred during the dynamic workflow.",
            details=f"{e}"
        )
        traceback.print_exc()
        return (None, None)

### **2.2 Input Validation Helpers**
**The Goal:**
To act as a "safety check" for your inputs. These functions ensure that the dates you request exist on the calendar and that your coordinates are valid latitude/longitude numbers.

**How to use this cell:**
* **Run the cell.** (No user input required).

In [ ]:
# --- 2.2: Input Validation Helpers ---

def validate_and_normalize_year(year_input):
    """Validates and normalizes a year input to a four-digit integer.

    Args:
        year_input (int or str): The year to validate (e.g., 2022 or "2022").

    Returns:
        int: The validated year as an integer.

    Raises:
        ValueError: If the input cannot be resolved to a valid year format
            or is outside the acceptable range (1851-current year).
    """
    try:
        year = int(year_input)
    except (ValueError, TypeError):
        raise ValueError(f"'{year_input}' is not a valid year format.")

    current_year = datetime.now().year
    if not (1851 <= year <= current_year):
        raise ValueError(f"Year must be between 1851 and {current_year}.")
    return year


def validate_and_normalize_state(state_input):
    """Validates and normalizes a state name to a capitalized string.

    Args:
        state_input (str): The state name to validate.

    Returns:
        str: The cleaned and capitalized state name.

    Raises:
        ValueError: If the input is not a non-empty string.
    """
    if not isinstance(state_input, str) or not state_input:
        raise ValueError("State must be a non-empty string.")
    return state_input.strip().capitalize()


def validate_and_normalize_datetime(date_str, date_format="%m/%d/%Y %H:%M"):
    """Validates a date string and converts it to a datetime object.

    Args:
        date_str (str): The date string to validate.
        date_format (str): The format of the date string.

    Returns:
        datetime: The parsed datetime object.

    Raises:
        ValueError: If the date string does not match the specified format.
    """
    try:
        return datetime.strptime(date_str, date_format)
    except ValueError:
        raise ValueError(f"'{date_str}' is not a valid date format. Please use '{date_format}'.")


def validate_lat_lon(lat, lon):
    """Validates that latitude and longitude are within their valid ranges.

    Args:
        lat (float): The latitude to validate (-90 to 90).
        lon (float): The longitude to validate (-180 to 180).

    Returns:
        tuple: A tuple containing the validated (lat, lon).

    Raises:
        ValueError: If latitude or longitude is out of its valid range.
    """
    if not (-90 <= lat <= 90):
        raise ValueError(f"Latitude must be between -90 and 90, but got {lat}.")
    if not (-180 <= lon <= 180):
        raise ValueError(f"Longitude must be between -180 and 180, but got {lon}.")
    return lat, lon


def construct_and_validate_datetime(year, month, day, hour=0, minute=0):
    """
    Constructs a datetime object from individual components and validates them.

    Args:
        year (int): The four-digit year.
        month (int): The month (1-12).
        day (int): The day (1-31).
        hour (int): The hour (0-23).
        minute (int): The minute (0-59).

    Returns:
        datetime: A valid datetime object.

    Raises:
        ValueError: If the provided components do not form a valid date
            (e.g., month=13, day=32).
    """
    try:
        # The datetime constructor automatically validates the components
        return datetime(year, month, day, hour, minute)
    except ValueError as e:
        # Re-raise the error with a more user-friendly message
        raise ValueError(
            f"Invalid date components provided: Year={year}, Month={month}, "
            f"Day={day}, Hour={hour}, Minute={minute}. Details: {e}"
        )

### **2.3 Geospatial & Analysis Helpers**
**The Goal:**
To perform the core math of the notebook. The key function here calculates the **"Nearest Neighbor"**—finding the single closest data grid point to your specific location.

**"Nearest Neighbor"**:
* It takes the precise geographic point (Lat/Lon) you want.
* It performs the mathematical distance calculation (Haversine formula).
* It identifies the closest fixed data point (grid cell) in the raw precipitation data.
These helpers also handle slicing and organizing the resulting time-series into a clear, usable format. 

**How to use this cell:**
* **Run the cell.** (No user input required).

In [ ]:
# --- 2.3: Geospatial & Analysis Helpers ---
HAWAII_QPE_SHIFT_DATE = datetime(2022, 4, 25, 17)

def calculate_haversine_distance(lat1, lon1, lat2, lon2, unit='km'):
    """
    Calculates the great-circle distance between two points on the earth.

    Args:
        lat1, lon1: Latitude and longitude of the first point.
        lat2, lon2: Latitude and longitude of the second point.
        unit (str): The desired output unit ('km' or 'miles').

    Returns:
        float: The distance in the specified unit.
    """
    R = 6371  # Radius of the Earth in kilometers

    lat1_rad = radians(lat1)
    lon1_rad = radians(lon1)
    lat2_rad = radians(lat2)
    lon2_rad = radians(lon2)

    dlon = lon2_rad - lon1_rad
    dlat = lat2_rad - lat1_rad

    a = sin(dlat / 2)**2 + cos(lat1_rad) * cos(lat2_rad) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    distance_km = R * c

    if unit.lower() == 'miles':
        return distance_km * 0.621371
    return distance_km

def snap_to_grid(target_val, res, offset):
    """
    Core calculation for snapping any coordinate to a fixed grid based on a specific resolution and offset.
    """
    return round(round((target_val - offset) / res) * res + offset, 3)

def calculate_grid_cell_offline(lat, lon, region, s3_path = "", target_dt = None):
    """
    Calculates the nearest MRMS grid cell mathematically.
    This bypasses the need to download state-wide coordinate columns from S3.

    Args:
        lat (float): The target latitude.
        lon (float): The target longitude.
        region (str): The state/region name to determine grid offsets.

    Returns:
        tuple: A tuple containing the calculated (grid_lat, grid_lon).
    """
    try:
        # Determine dataset type directly from the S3 path string
        is_ffg = "FLASH_QPE_FFGMAX" in str(s3_path).upper() or "FFG" in str(s3_path).upper()

        if region == "Hawaii":
            if is_ffg:
                # Hawaii QPE/FFG: Standard lat spacing, drifting GRIB2 longitudinal step
                grid_lat = snap_to_grid(lat, res=0.005, offset=0.0)
                grid_lon = snap_to_grid(lon, res=0.0049980727, offset=-163.994996)
            else:
                # Hawaii QPE: Underwent era shift on April 25, 2022
                if target_dt and target_dt >= HAWAII_QPE_SHIFT_DATE:
                    grid_lat = snap_to_grid(lat, res=0.005, offset=0.003)
                    grid_lon = snap_to_grid(lon, res=0.005, offset=0.002)
                else:
                    grid_lat = snap_to_grid(lat, res=0.005, offset=0.0)
                    grid_lon = snap_to_grid(lon, res=0.005, offset=0.0)

        elif region == "Guam":
            if is_ffg:
                # Guam QPE/FFG: Lat uses 0.0 offset, Lon uses 0.003 offset
                grid_lat = snap_to_grid(lat, res=0.005, offset=0.0)
                grid_lon = snap_to_grid(lon, res=0.005, offset=0.003)
            else:
                # Guam QPE: Both axes use 0.003 offset
                grid_lat = snap_to_grid(lat, res=0.005, offset=0.003)
                grid_lon = snap_to_grid(lon, res=0.005, offset=0.003)

        else:
            # CONUS, Alaska, Puerto Rico (Standard 0.01 grid)
            grid_lat = snap_to_grid(lat, res=0.01, offset=0.005)
            grid_lon = snap_to_grid(lon, res=0.01, offset=0.005)

        return grid_lat, grid_lon
    except Exception as e:
        warnings.warn(f"  Warning: Grid calculation failed. Using raw coordinates. Error: {e}", UserWarning)
        return round(lat, 3), round(lon, 3)

def find_nearest_neighbors(locations_dict, region, s3_path="", start_dt=None, end_dt=None):
    """
    Finds nearest grid cell mathematically for each location.
    Generates dual-era coordinates if query spans the April 25, 2022 Hawaii QPE shift.
    """
    try:
        if not locations_dict:
            print("⚠️ Warning: Locations dictionary is empty.")
            return None, None

        is_ffg = "FLASH_QPE_FFGMAX" in str(s3_path).upper() or "FFG" in str(s3_path).upper()
        is_hawaii_qpe = (region == "Hawaii") and not is_ffg
        spans_era = is_hawaii_qpe and (start_dt and end_dt and start_dt < HAWAII_QPE_SHIFT_DATE and end_dt >= HAWAII_QPE_SHIFT_DATE)

        nearest_points = []
        feedback_list = []

        for name, coords in locations_dict.items():
            if (not isinstance(coords, dict) or
                'latitude' not in coords or
                'longitude' not in coords):
                print(f"⚠️ Warning: Skipping '{name}' due to malformed coordinate dictionary.")
                continue

            lat, lon = coords['latitude'], coords['longitude']
            validate_lat_lon(lat, lon)

            if spans_era:
                # Append BOTH pre-shift and post-shift grid points for this site
                old_lat, old_lon = calculate_grid_cell_offline(lat, lon, region, s3_path, start_dt)
                new_lat, new_lon = calculate_grid_cell_offline(lat, lon, region, s3_path, end_dt)

                nearest_points.append({"location_name": name, "latitude": old_lat, "longitude": old_lon})
                nearest_points.append({"location_name": name, "latitude": new_lat, "longitude": new_lon})

                distance_km = calculate_haversine_distance(lat, lon, new_lat, new_lon)

                feedback_list.append({
                    "location_name": f"{name} (Date Shift)",
                    "target_lat": lat,
                    "target_lon": lon,
                    "selected_lat": new_lat,
                    "selected_lon": new_lon,
                    "distance_km": distance_km
                })
            else:
                ref_dt = start_dt or datetime.now()
                nearest_lat, nearest_lon = calculate_grid_cell_offline(lat, lon, region, s3_path, ref_dt)
                distance_km = calculate_haversine_distance(lat, lon, nearest_lat, nearest_lon)

                nearest_points.append({"location_name": name, "latitude": nearest_lat, "longitude": nearest_lon})
                feedback_list.append({
                    "location_name": name,
                    "target_lat": lat,
                    "target_lon": lon,
                    "selected_lat": nearest_lat,
                    "selected_lon": nearest_lon,
                    "distance_km": distance_km
                })

        return pl.DataFrame(nearest_points), feedback_list

    except Exception as e:
        print(f"❌ Error: Failed to find nearest neighbors. \n   Details: {e}")
        traceback.print_exc()
        return None, None


def get_timeseries_for_geographic_point(precip_lazy_df, lat, lon, region, start_dt, end_dt, s3_path=""):
    """
    Extracts a single location time series using lazy evaluation and mathematical grid mapping.
    Handles era-spanning queries smoothly with exact conditional lat/lon assignment.
    """
    try:
        is_ffg = "FLASH_QPE_FFGMAX" in str(s3_path).upper() or "FFG" in str(s3_path).upper()
        is_hawaii_qpe = (region == "Hawaii") and not is_ffg
        spans_era = is_hawaii_qpe and (start_dt < HAWAII_QPE_SHIFT_DATE and end_dt >= HAWAII_QPE_SHIFT_DATE)

        if spans_era:
            old_lat, old_lon = calculate_grid_cell_offline(lat, lon, region, s3_path, start_dt)
            new_lat, new_lon = calculate_grid_cell_offline(lat, lon, region, s3_path, end_dt)

            nearest_lat, nearest_lon = new_lat, new_lon
            distance_km = calculate_haversine_distance(lat, lon, nearest_lat, nearest_lon)

            filter_old = (pl.col("datetime") < HAWAII_QPE_SHIFT_DATE) & (pl.col("latitude") == old_lat) & (pl.col("longitude") == old_lon)
            filter_new = (pl.col("datetime") >= HAWAII_QPE_SHIFT_DATE) & (pl.col("latitude") == new_lat) & (pl.col("longitude") == new_lon)

            filtered_lazy = precip_lazy_df.filter(
                (filter_old | filter_new) & pl.col("datetime").is_between(start_dt, end_dt)
            )
        else:
            nearest_lat, nearest_lon = calculate_grid_cell_offline(lat, lon, region, s3_path, start_dt)
            distance_km = calculate_haversine_distance(lat, lon, nearest_lat, nearest_lon)

            filtered_lazy = precip_lazy_df.filter(
                (pl.col("latitude") == nearest_lat) &
                (pl.col("longitude") == nearest_lon) &
                (pl.col("datetime").is_between(start_dt, end_dt))
            )

        feedback_info = {
            "target_lat": lat,
            "target_lon": lon,
            "selected_lat": nearest_lat,
            "selected_lon": nearest_lon,
            "distance_km": distance_km
        }

        final_timeseries_df = filtered_lazy.collect()

        # In-fill timeline gaps (dry hours)
        datetime_range = pl.datetime_range(
            start_dt, end_dt, interval="1h", closed="both",
            time_unit="ns", eager=True)
        datetime_range_df = pl.DataFrame({"datetime": datetime_range})

        if not final_timeseries_df.is_empty():
            final_timeseries_df = final_timeseries_df.with_columns(
                pl.col("datetime").cast(pl.Datetime("ns"))
            )

        final_df = final_timeseries_df.join(datetime_range_df, on="datetime", how="full", coalesce=True)

        # Assign coordinates dynamically based on date era
        if spans_era:
            final_df = final_df.with_columns(
                pl.when(pl.col("datetime") < HAWAII_QPE_SHIFT_DATE).then(old_lat).otherwise(new_lat).alias("latitude"),
                pl.when(pl.col("datetime") < HAWAII_QPE_SHIFT_DATE).then(old_lon).otherwise(new_lon).alias("longitude")
            )
        else:
            final_df = final_df.with_columns(
                pl.col("latitude").fill_null(nearest_lat),
                pl.col("longitude").fill_null(nearest_lon)
            )

        return final_df.select(
            ["datetime", "observation", "latitude", "longitude"]
        ).sort("datetime"), feedback_info

    except Exception as e:
        print(f"❌ Error during geographic time series extraction. \n   Details: {e}")
        traceback.print_exc()
        return None, None


def extract_and_pivot_timeseries(precip_lazy_df, nearest_points_df, start_dt,
                                 end_dt, value_col="observation"):
    """
    Extracts time-series data lazily, filters by datetime, and pivots into a wide format.
    """
    try:
        nearest_points_df = nearest_points_df.with_columns(
            pl.col("latitude").cast(pl.Float64),
            pl.col("longitude").cast(pl.Float64)
        )

        time_filtered_lazy = precip_lazy_df.join(
            nearest_points_df.lazy(), on=["latitude", "longitude"], how="inner"
        ).filter(
            (pl.col("datetime") >= start_dt) & (pl.col("datetime") <= end_dt)
        )

        time_filtered_df = time_filtered_lazy.collect()

        if not time_filtered_df.is_empty() and value_col not in time_filtered_df.columns:
            raise ValueError(
                f"The specified value column '{value_col}' was not found in the DataFrame. "
                f"Available columns are: {time_filtered_df.columns}"
            )

        if not time_filtered_df.is_empty():
            pivoted_df = time_filtered_df.pivot(
                values=value_col,
                index="datetime",
                on="location_name"
            ).sort("datetime")
        else:
            schema = {'datetime': pl.Datetime}
            pivoted_df = pl.DataFrame(schema=schema)

        datetime_range = pl.datetime_range(
            start_dt, end_dt, interval="1h", closed="both", time_unit="ns", eager=True
        )
        full_range_df = pl.DataFrame({"datetime": datetime_range})

        if not pivoted_df.is_empty():
            pivoted_df = pivoted_df.with_columns(
                pl.col("datetime").cast(pl.Datetime("ns"))
            )

        final_df = full_range_df.join(pivoted_df, on="datetime", how="left")

        # Get unique location names to prevent duplicate columns if dual-era rows were joined
        all_location_names = nearest_points_df['location_name'].unique(maintain_order=True).to_list()
        for loc_name in all_location_names:
            if loc_name not in final_df.columns:
                final_df = final_df.with_columns(
                    pl.lit(None, dtype=pl.Float64).alias(loc_name)
                )

        final_df = final_df.select(['datetime'] + all_location_names)
        return final_df

    except Exception as e:
        print(f"❌ Error during data extraction and pivoting: \n   Details: {e}")
        traceback.print_exc()
        return pl.DataFrame()


def run_single_location_analysis(s3_path, s3_filesystem, lat, lon, start_dt, end_dt,
                               distance_threshold=25.0):
    """
    High-level workflow function to run full single-location analysis.
    """
    try:
        match = re.search(r"state=([^/]+)", s3_path)
        region = match.group(1) if match else "Washington"

        if region == "Alaska" and "FLASH_QPE_FFGMAX" in s3_path:
             log_user_error(
                 "Bypassed Alaska FLASH_QPE_FFGMAX to prevent runtime crash.",
                 hint="This is a known data-quality issue in the S3 bucket."
             )
             return None, None

        print(f"Building query plan for: {s3_path}...")
        lazy_precip_df = retrieve_data_from_s3(s3_path, s3_filesystem)

        print(f"Extracting time series for (Lat: {lat}, Lon: {lon}) mathematically...")
        single_loc_df, feedback = get_timeseries_for_geographic_point(
            lazy_precip_df,
            lat,
            lon,
            region,
            start_dt,
            end_dt,
            s3_path=s3_path
        )

        if feedback:
            print("\n--- Nearest Grid Cell Selection ---")
            print(f"Target Location:         "
                  f"Lat={feedback['target_lat']:.3f}, "
                  f"Lon={feedback['target_lon']:.3f}")
            print(f"Closest Grid Cell Found: "
                  f"Lat={feedback['selected_lat']:.3f}, "
                  f"Lon={feedback['selected_lon']:.3f}")
            print(f"Distance (km): {feedback['distance_km']:.1f}")

            if feedback['distance_km'] > distance_threshold:
                print(f" - ⚠️ WARNING: Target is >{distance_threshold} km "
                      "from the nearest available data point.")
            print("-----------------------------------\n")

        if single_loc_df is None or single_loc_df.is_empty():
            print("\nWarning: No data found for this location and time range.")
        else:
            print(f"\nSuccessfully extracted {len(single_loc_df)} "
                  "time series records.")
            null_count = single_loc_df.select(
                pl.col('observation').is_null().sum()
            )[0, 0]
            print(f"  (Including {null_count} records with null "
                  "observations - i.e., missing hours).")

        return single_loc_df, feedback

    except Exception as e:
        print(f"❌ An unexpected error occurred during data analysis.")
        traceback.print_exc()
        return None, None


def run_multi_location_analysis(s3_path, s3_filesystem, locations_dict,
                                start_dt, end_dt, value_col="observation",
                                distance_threshold=25.0):
    """
    High-level workflow function to run full multi-location analysis.
    Passes both start_dt and end_dt down to support era-spanning queries.
    """
    try:
        match = re.search(r"state=([^/]+)", s3_path)
        region = match.group(1) if match else "Washington"

        if region == "Alaska" and "FLASH_QPE_FFGMAX" in s3_path:
             log_user_error(
                 "Bypassed Alaska FLASH_QPE_FFGMAX to prevent runtime crash.",
                 hint="This is a known data-quality issue in the S3 bucket."
             )
             return None, None

        print(f"Building query plan for: {s3_path}...")
        lazy_precip_df = retrieve_data_from_s3(s3_path, s3_filesystem)

        print("Finding nearest grid points for all target locations mathematically...")
        nearest_points_df, feedback_list = find_nearest_neighbors(
            locations_dict, region, s3_path=s3_path, start_dt=start_dt, end_dt=end_dt
        )

        if nearest_points_df is None or not feedback_list:
            raise RuntimeError("Failed to calculate nearest neighbors.")

        print("\n--- Nearest Neighbor Analysis ---")
        header = f"{'Location':<20} | {'Target Coords':<22} | " \
                 f"{'Selected Coords':<22} | {'Distance (km)':<15}"
        print(header)
        print("-" * len(header))

        distance_warning = False
        for item in feedback_list:
            loc = item['location_name']
            t_lat, t_lon = item['target_lat'], item['target_lon']
            s_lat, s_lon = item['selected_lat'], item['selected_lon']
            dist = item['distance_km']

            print(f"{loc:<20} | ({t_lat: >8.4f}, {t_lon: >8.4f}) | "
                  f"({s_lat: >8.4f}, {s_lon: >8.4f}) | {dist:<15.2f}")

            if dist > distance_threshold:
                distance_warning = True

        if distance_warning:
            print(f"\n⚠️ WARNING: One or more locations are farther than "
                  f"{distance_threshold} km")
            print("   from their nearest grid point. This may affect data accuracy.")
        else:
            print("\n✅ All locations are within the distance threshold.")

        print("\nExtracting and pivoting time series data...")
        pivoted_df = extract_and_pivot_timeseries(
            lazy_precip_df,
            nearest_points_df,
            start_dt,
            end_dt,
            value_col=value_col
        )

        if pivoted_df is None or pivoted_df.is_empty():
            print("Warning: No data found for the specified locations and time range.")
        else:
            print(f"Successfully pivoted {len(pivoted_df)} time series records.")

        return pivoted_df, nearest_points_df

    except RuntimeError as e:
        raise e
    except Exception as e:
        print(f"❌ An unexpected error occurred during multi-location analysis.")
        traceback.print_exc()
        return None, None

### **2.4 Plotting Utility Helpers**
**The Goal:**
To dynamically calculate chart settings. Specifically, this logic determines how many tick marks to show on the X-axis (Time) so the dates are readable regardless of whether you plot 2 days or 60 days.

**How to use this cell:**
* **Run the cell.** (No user input required).

In [ ]:
# --- 2.4: Plotting Utility Helpers ---

def simple_word_wrap(text, wrap_width=80):
    """Wraps a given text at word breaks to a specified width.

    Args:
        text (str): The text to be wrapped.
        wrap_width (int): The maximum width of each line.

    Returns:
        str: The wrapped text with newlines.
    """
    return '\n'.join(textwrap.wrap(text, width=wrap_width))


def _determine_xaxis_ticks(start_dt, end_dt, params):
    """
    Determines appropriate major/minor locators and formatters for the x-axis
    based on duration and user overrides.
    - None in params: Explicitly disable the locator/formatter.
    - "" or 'default' or key omitted: Use calculated default for that setting.
    - Other string: Attempt to parse as override for that setting.

    Args:
        start_dt (datetime): The start datetime of the plot range.
        end_dt (datetime): The end datetime of the plot range.
        params (dict): Dictionary possibly containing user overrides.

    Returns:
        tuple: (final_major_locator, final_major_formatter,
                final_minor_locator, final_minor_formatter)
    """
    try:
        # --- Calculate Duration ---
        duration_days = None
        if start_dt and end_dt:
            duration = end_dt - start_dt
            duration_days = duration.total_seconds() / (24 * 3600) # Use float for precision

        # --- Get User Overrides  ---
        _not_set = object()
        user_major_interval = params.get('x_major_tick_interval', _not_set)
        user_minor_interval = params.get('x_minor_tick_interval', _not_set)
        user_major_format = params.get('x_tick_label_format', _not_set)
        user_minor_format = params.get('x_minor_tick_label_format', _not_set)

        # --- Determine Default Locators and Formatters ---
        default_major_locator = None
        default_major_formatter = None
        default_minor_locator = None
        default_minor_formatter = None

        if duration_days is not None:
            # --- 24-Hour Daily Plots ---
            if duration_days <= 1.1:
                default_major_locator = mdates.HourLocator(interval=4)
                default_major_formatter = mdates.DateFormatter('%m-%d %H:%M')
                default_minor_locator = mdates.HourLocator(interval=1)
                default_minor_formatter = mdates.DateFormatter("%H")

            # --- 2-4 Day Plots ---
            elif duration_days <= 4:
                default_major_locator = mdates.AutoDateLocator()
                default_major_formatter = mdates.AutoDateFormatter(default_major_locator)
                default_minor_locator = mdates.HourLocator(byhour=[0,6,12,18])
                default_minor_formatter = mdates.DateFormatter("%H")

            # --- 4-8 Day Plots ---
            elif duration_days <= 8:
                default_major_locator = mdates.DayLocator(interval=1)
                default_major_formatter = mdates.DateFormatter('%Y-%m-%d')
                default_minor_locator = mdates.HourLocator(byhour=[0,6,12,18])
                default_minor_formatter = mdates.DateFormatter("%H")

            # --- ~14-Day Composite Plots ---
            elif duration_days <= 14:
                default_major_locator = mdates.DayLocator(interval=1)
                default_major_formatter = mdates.DateFormatter('%Y-%m-%d')
                default_minor_locator = mdates.HourLocator(byhour=[0,6,12,18])
                default_minor_formatter = ticker.NullFormatter() # No label

            # --- ~20-60 Day Plots ---
            elif duration_days <= 60:
                default_major_locator = mdates.DayLocator(interval=4) # 4-day major
                default_major_formatter = mdates.DateFormatter('%Y-%m-%d')
                default_minor_locator = mdates.DayLocator(interval=1) # 1-day minor
                default_minor_formatter = mdates.DateFormatter('%m-%d')

            # --- > 60 Day Plots ---
            else:
                default_major_locator = mdates.AutoDateLocator()
                default_major_formatter = mdates.AutoDateFormatter(default_major_locator)
                default_minor_locator = ticker.AutoMinorLocator()
                default_minor_formatter = ticker.NullFormatter()

        # --- Final Fallback if all else fails ---
        if not default_major_locator: default_major_locator = mdates.AutoDateLocator()
        if not default_major_formatter: default_major_formatter = mdates.AutoDateFormatter(default_major_locator)
        if not default_minor_locator: default_minor_locator = ticker.AutoMinorLocator()
        if not default_minor_formatter: default_minor_formatter = ticker.NullFormatter()
        # --- END Default Logic ---


        # --- Initialize final objects with defaults ---
        final_major_locator = default_major_locator
        final_major_formatter = default_major_formatter
        final_minor_locator = default_minor_locator
        final_minor_formatter = default_minor_formatter

        # --- Process User Overrides ---

        # --- Major Locator ---
        if user_major_interval is _not_set or user_major_interval in ["", "default"]:
            pass # Use default
        elif user_major_interval is None: # Explicitly disable
            final_major_locator = ticker.NullLocator()
            final_major_formatter = ticker.NullFormatter() # Disable formatter too
        else: # Attempt to parse and apply interval string
            try:
                # Regex to find interval and unit (e.g., "6H", "1D")
                match = re.match(r"(\d+)\s*([HDWM])", user_major_interval.upper())
                if not match: raise ValueError("Format error")

                interval = int(match.group(1))
                unit = match.group(2)

                locator = None
                if unit == 'H': locator = mdates.HourLocator(interval=interval)
                elif unit == 'D': locator = mdates.DayLocator(interval=interval)
                elif unit == 'W': locator = mdates.WeekdayLocator(interval=interval)
                elif unit == 'M': locator = mdates.MonthLocator(interval=interval)
                else: raise ValueError("Unknown unit")

                if locator: final_major_locator = locator
            except Exception as e:
                warnings.warn(f"  Warning: Invalid major interval format '{user_major_interval}'. Using default. Error: {e}", UserWarning)

        # --- Minor Locator ---
        if not isinstance(final_major_locator, ticker.NullLocator):
            if user_minor_interval is _not_set or user_minor_interval in ["", "default"]:
                pass # Use default
            elif user_minor_interval is None: # Explicitly disable
                final_minor_locator = ticker.NullLocator()
                final_minor_formatter = ticker.NullFormatter() # Also disable formatter
            else: # Attempt to parse and apply interval string
                try:
                    match = re.match(r"(\d+)\s*([HD])", user_minor_interval.upper())
                    if not match: raise ValueError("Format error")

                    interval = int(match.group(1))
                    unit = match.group(2)

                    locator = None
                    if unit == 'H':
                        if interval > 0 and 24 % interval == 0:
                            hours_list = list(range(0, 24, interval))
                            locator = mdates.HourLocator(byhour=hours_list)
                        else:
                            warnings.warn(f"  Warning: Hourly interval {interval} invalid. Reverting to default.", UserWarning)
                            locator = default_minor_locator
                    elif unit == 'D':
                        if interval == 1:
                            locator = mdates.DayLocator(interval=interval)
                        else:
                            warnings.warn(f"  Warning: Day interval > 1 invalid. Reverting to default.", UserWarning)
                            locator = default_minor_locator
                    else: raise ValueError("Unknown unit")

                    if locator: final_minor_locator = locator
                except Exception as e:
                    warnings.warn(f"  Warning: Invalid minor interval format/value '{user_minor_interval}'. Using default. Error: {e}", UserWarning)
        else: # Major locator was disabled, so disable minor too
            final_minor_locator = ticker.NullLocator()
            final_minor_formatter = ticker.NullFormatter()

        # --- Major Formatter ---
        if not isinstance(final_major_locator, ticker.NullLocator):
            if user_major_format is _not_set or user_major_format in ["", "default"]:
                pass # Use default
            elif user_major_format is None: # Explicitly disable
                final_major_formatter = ticker.NullFormatter()
            else: # Attempt to apply format string
                try:
                    formatter = mdates.DateFormatter(user_major_format)
                    final_major_formatter = formatter
                except Exception as e:
                    warnings.warn(f"  Warning: Invalid major format string '{user_major_format}'. Using default. Error: {e}", UserWarning)

        # --- Minor Formatter ---
        if not isinstance(final_minor_locator, ticker.NullLocator):
            if user_minor_format is _not_set or user_minor_format in ["", "default"]:
                pass # Use default
            elif user_minor_format is None: # Explicitly disable
                final_minor_formatter = ticker.NullFormatter()
            else: # Attempt to apply format string
                try:
                    formatter = mdates.DateFormatter(user_minor_format)
                    final_minor_formatter = formatter
                except Exception as e:
                    warnings.warn(f"  Warning: Invalid minor format string '{user_minor_format}'. Using default. Error: {e}", UserWarning)

        # --- Return the final objects ---
        return final_major_locator, final_major_formatter, final_minor_locator, final_minor_formatter

    except Exception as e:
        print(f"❌ Error in _determine_xaxis_ticks: {e}")
        traceback.print_exc()
        return None, None, None, None

### **2.5 Primary Plotting Functions**
**The Goal:**
To define the visual style of the charts. This section contains the code that actually draws the bars, stems, and labels for the three main plot types. These specialized functions implement the styling, colors, and analytical alt text (textual descriptions for accessibility) needed to create the final visualizations.

**How to use this cell:**
* **Run the cell.** (No user input required).

In [ ]:
# --- 2.5: Primary Plotting Functions ---

def plot_single_location_timeseries(timeseries_df, params={}, ax=None):
    """
    Generates a bar chart for a single location, configuring x-axis ticks/labels
    by calling the _determine_xaxis_ticks helper function.

    Includes minor ticks on both axes and optimized x-tick label rotation.
    Ensures y-axis spans a consistent range if specified.

    Args:
        timeseries_df (pl.DataFrame): DataFrame with 'datetime', 'observation'.
        params (dict): Dictionary of plotting parameters.
                       - Requires 'start_datetime', 'end_datetime'.
                       - Optional: 'global_y_max', 'lat', 'lon'.
                       - Optional: 'x_major_tick_interval', 'x_minor_tick_interval',
                                   'x_tick_label_format', 'x_minor_tick_label_format'.
        ax (matplotlib.axes.Axes, optional): Axes to draw on. Defaults to None.

    Returns:
        dict: Dictionary containing the figure object and alt text.
    """
    # --- Input Validation ---
    if timeseries_df is None:
        print("Warning: Cannot generate plot because no data was provided.")
        return {'figure': None, 'alt_text': "Plot could not be generated."}

    create_figure = ax is None

    try:
        # --- Data Preparation ---
        # --- Data Integrity: We filter for plotting, NOT for analysis ---
        # This is a key pattern: we respect data integrity by
        # not changing NaNs to 0s in the source data. We just plot non-nulls.
        plot_df = timeseries_df.filter(pl.col("observation").is_not_null())
        start_dt = params.get('start_datetime')
        end_dt = params.get('end_datetime')

        # --- Plot Generation ---
        if create_figure:
            fig, current_ax = plt.subplots(
                figsize=params.get('figsize', (12, 5)),
                dpi=params.get('fig_dpi', None))
            fig.subplots_adjust(bottom=0.2)
        else:
            current_ax = ax
            fig = current_ax.get_figure()

        # Add styled major/minor grid for BOTH axes
        current_ax.grid(True, zorder=0, which='major', axis='y', linestyle='-')
        current_ax.grid(True, zorder=0, which='minor', axis='y', linestyle=':')
        current_ax.grid(True, zorder=0, which='major', axis='x', linestyle='--')
        current_ax.grid(True, zorder=0, which='minor', axis='x', linestyle=':')

        if not plot_df.is_empty():
            current_ax.bar(
                plot_df["datetime"],
                plot_df["observation"],
                width=params.get('bar_width', 0.04),
                color=params.get('color', '#0076D6'),
                zorder=3
            )

        # --- Plot Styling and Finalization ---
        tick_spacing = params.get('y_tick_spacing', 0.1)

        # --- Y-AXIS SCALING ---
        if 'global_y_max' in params:
            max_val = params['global_y_max']
        else:
            max_val_local = timeseries_df['observation'].max()
            max_val = (max_val_local if max_val_local is not None
                       and max_val_local > 0 else 0)

        if max_val > 0:
            # Calculate the ceiling limit based on tick spacing
            # (e.g., 0.29 with 0.1 spacing -> 0.30)
            max_y_limit = (max_val // tick_spacing + 1) * tick_spacing

            # Generate ticks up to the clean limit
            yticks = [i * tick_spacing
                      for i in range(int(max_y_limit / tick_spacing) + 1)]
            current_ax.set_yticks(yticks)

            # Set the limit *exactly* to the max tick
            current_ax.set_ylim(bottom=0, top=max_y_limit)
        else:
            # If max is 0, just show one tick spacing
            max_y_limit = tick_spacing
            current_ax.set_yticks([0, max_y_limit])
            current_ax.set_ylim(bottom=0, top=max_y_limit)

        current_ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())

        # Handle Title creation
        if 'title' in params:
            title = params['title']
        else:
            lat = params.get('lat')
            lon = params.get('lon')
            if isinstance(lat, (int, float)) and isinstance(lon, (int, float)):
                title = f"QPE for Lat: {lat:.4f}, Lon: {lon:.4f}"
            else:
                location_name = params.get('location_name', 'Selected Location')
                title = f"QPE for {location_name}"
        current_ax.set_title(title)

        # Set Labels
        if create_figure or params.get('set_xlabel_subplot', False):
             current_ax.set_xlabel(params.get('xlabel', "Date"))
        current_ax.set_ylabel(params.get('ylabel', "QPE (in)"))

        # Apply Consistent Time Frame
        if start_dt and end_dt:
            current_ax.set_xlim(
                mdates.date2num(start_dt), mdates.date2num(end_dt))

        # Call the helper function to get the final locators and formatters
        final_major_locator, final_major_formatter, \
        final_minor_locator, final_minor_formatter = _determine_xaxis_ticks(
            start_dt, end_dt, params
            )

        # Apply the determined objects to the axis
        if final_major_locator:
            current_ax.xaxis.set_major_locator(final_major_locator)
        if final_major_formatter:
            current_ax.xaxis.set_major_formatter(final_major_formatter)
        if final_minor_locator:
            current_ax.xaxis.set_minor_locator(final_minor_locator)
        if final_minor_formatter:
            current_ax.xaxis.set_minor_formatter(final_minor_formatter)

        # Apply Tick Sizing/Styling from Params ---
        # Apply parameters to MAJOR ticks
        current_ax.tick_params(
            axis='x',
            which='major',
            direction=params.get('x_tick_direction', 'out'),
            length=params.get('x_major_tick_length', 6),
            width=params.get('x_major_tick_width', 1.0),
            labelsize=params.get('x_major_tick_labelsize', 'medium')
        )
        # Apply parameters to MINOR ticks
        current_ax.tick_params(
            axis='x',
            which='minor',
            direction=params.get('x_tick_direction', 'out'),
            length=params.get('x_minor_tick_length', 3),
            width=params.get('x_minor_tick_width', 0.6),
            labelsize=params.get('x_minor_tick_labelsize', 'x-small'),
            labelbottom=True
        )

        # Apply rotation using autofmt_xdate for final layout adjustment
        fig.autofmt_xdate(rotation=45, ha='right')


        # 5. --- Alt Text Generation ---

        # --- New Details for Accessibility ---
        # We use the original timeseries_df for the total.
        # polars .sum() correctly handles nulls (treats them as 0 for sum)
        total_precip = timeseries_df['observation'].sum()

        # Format date range
        start_date_str = start_dt.strftime('%Y-%m-%d') if start_dt else 'N/A'
        end_date_str = end_dt.strftime('%Y-%m-%d') if end_dt else 'N/A'
        date_range_str = f"from {start_date_str} to {end_date_str}"

        # Get min/max from the plotted data (plot_df)
        min_val_obs = plot_df['observation'].min()
        max_val_obs = plot_df['observation'].max()

        if plot_df.is_empty():
            alt_text_string = (
                f"Bar chart titled '{title}'. Shows data {date_range_str}. "
                "No precipitation data was observed."
            )
        else:
            alt_text_string = (
                f"Bar chart titled '{title}'. "
                f"Shows hourly precipitation {date_range_str}. "
                f"Total precipitation: {total_precip:.2f} inches. "
                f"Hourly values range from {min_val_obs:.2f} to "
                f"{max_val_obs:.2f} inches."
            )

        if create_figure:
            print(f"\nAlt text: {simple_word_wrap(alt_text_string)}")


        return {'figure': fig, 'alt_text': alt_text_string}

    except Exception as e:
        print(f"❌ Error during single-location plot: \n   Details: {e}")
        traceback.print_exc()
        if create_figure and 'fig' in locals():
            plt.close(fig)
        return {'figure': None, 'alt_text': "Plot could not be generated."}



def plot_multi_location_stem_plot(multi_timeseries_df, start_dt, end_dt, params={}):
    """
    Generates a dynamic grouped stem plot, configuring x-axis ticks/labels
    by calling the _determine_xaxis_ticks helper function and applying detailed
    tick size/style parameters.

    Includes minor ticks on both axes and optimized x-tick label rotation.
    Legend positioned below the plot. Uses unique markers per location.

    Args:
        multi_timeseries_df (pl.DataFrame): The pivoted time series data.
        start_dt (datetime): The start datetime of the full plot range.
        end_dt (datetime): The end datetime of the full plot range.
        params (dict): Dictionary of plotting parameters, including optional
                       x-axis tick/label controls and size/style settings.

    Returns:
        dict: Dictionary containing the figure object and alt text.
    """
    # --- Input Validation ---
    if multi_timeseries_df is None or multi_timeseries_df.is_empty():
        print("Warning: Cannot generate plot because no data was provided.")
        return {'figure': None, 'alt_text': "Plot could not be generated."}

    try:
        # --- Data Preparation ---
        location_names = params.get(
            'location_order',
            [col for col in multi_timeseries_df.columns if col != 'datetime']
        )
        num_locations = len(location_names)

        # --- Data Integrity: Filter for plotting only ---
        plot_df = multi_timeseries_df.filter(
            pl.any_horizontal(pl.col(location_names).is_not_null())
        )

        if plot_df.is_empty():
            print("Info: No precipitation data to plot in the selected time range.")
            return {'figure': None, 'alt_text': "No data to plot."}

        date_nums = mdates.date2num(plot_df["datetime"].to_list())

        # --- Plot Generation ---
        fig, ax = plt.subplots(
            figsize=params.get('figsize', (12, 8)),
            dpi=params.get('fig_dpi', None))
        fig.subplots_adjust(bottom=0.3) # Initial room for legend/labels

        # Add styled major/minor grid for BOTH axes
        ax.grid(True, zorder=0, which='major', axis='y', linestyle='-')
        ax.grid(True, zorder=0, which='minor', axis='y', linestyle=':')
        ax.grid(True, zorder=0, which='major', axis='x', linestyle='--')
        ax.grid(True, zorder=0, which='minor', axis='x', linestyle=':')

        offset_width = params.get('stem_offset_width', 0.008)
        colors = params.get('colors', ['#0076D6', '#162E51', '#084fc9', '#707070'])
        locations_map = params.get('locations_map', {})
        markers = params.get('markers', ['o'] * num_locations)

        for i, loc_name in enumerate(location_names):
            if loc_name in plot_df.columns:
                offset = offset_width * (i - (num_locations - 1)) / 2
                coords = locations_map.get(loc_name, {})
                lat = coords.get('latitude')
                lon = coords.get('longitude')

                if isinstance(lat, (int, float)) and isinstance(lon, (int, float)):
                    legend_label = f"{loc_name} (Lat: {lat:.2f}, Lon: {lon:.2f})"
                else:
                    legend_label = f"{loc_name} (Coordinates N/A)"

                markerline, stemlines, baseline = ax.stem(
                    date_nums + offset, plot_df[loc_name],
                    label=legend_label,
                    basefmt=" "
                )

                current_marker = markers[i % len(markers)]
                plt.setp(stemlines, 'color', colors[i % len(colors)], 'zorder', 3)
                plt.setp(
                    markerline, marker=current_marker,
                    color=colors[i % len(colors)], zorder=3)

        # --- Plot Styling ---
        tick_spacing = params.get('y_tick_spacing', 0.1)

        all_values = plot_df.select(pl.all().exclude("datetime")).to_numpy().flatten()
        valid_values = all_values[~np.isnan(all_values)]
        if len(valid_values) > 0:
            max_val = valid_values.max()
            if max_val > 0:
                max_y_limit = (max_val // tick_spacing + 1) * tick_spacing
                # Ensure top limit is exactly the max tick
                yticks = [i * tick_spacing
                          for i in range(int(max_y_limit / tick_spacing) + 1)]
                ax.set_yticks(yticks)
                ax.set_ylim(bottom=0, top=max_y_limit)
            else:
                yticks = [i * tick_spacing for i in range(int(1 / tick_spacing) + 1)]
                ax.set_yticks(yticks)
                ax.set_ylim(bottom=0, top=1)
            ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
        else:
            yticks = [i * tick_spacing for i in range(int(1 / tick_spacing) + 1)]
            ax.set_yticks(yticks)
            ax.set_ylim(bottom=0, top=1)

        region_name = params.get('state', params.get('region', 'Selected Region'))
        default_title = f"QPE for {num_locations} locations in {region_name}"
        title = params.get('title', default_title)

        ax.set_title(title)
        ax.set_xlabel(params.get('xlabel', "Date"))
        ax.set_ylabel(params.get('ylabel', "QPE (in)"))

        # Apply Consistent Time Frame
        if start_dt and end_dt:
            ax.set_xlim(mdates.date2num(start_dt), mdates.date2num(end_dt))

        # --- X-AXIS TICK LOGIC using Helper ---
        #Determine final locators/formatters using the helper
        final_major_locator, final_major_formatter, \
        final_minor_locator, final_minor_formatter = _determine_xaxis_ticks(
            start_dt, end_dt, params)

        #Apply the determined objects
        if final_major_locator: ax.xaxis.set_major_locator(final_major_locator)
        if final_major_formatter: ax.xaxis.set_major_formatter(final_major_formatter)
        if final_minor_locator: ax.xaxis.set_minor_locator(final_minor_locator)
        if final_minor_formatter: ax.xaxis.set_minor_formatter(final_minor_formatter)

        # Apply Tick Sizing/Styling from Params
        ax.tick_params(
            axis='x', which='major',
            direction=params.get('x_tick_direction', 'out'),
            length=params.get('x_major_tick_length', 6),
            width=params.get('x_major_tick_width', 1.0),
            labelsize=params.get('x_major_tick_labelsize', 'medium')
        )
        ax.tick_params(
            axis='x', which='minor',
            direction=params.get('x_tick_direction', 'out'),
            length=params.get('x_minor_tick_length', 3),
            width=params.get('x_minor_tick_width', 0.6),
            labelsize=params.get('x_minor_tick_labelsize', 'x-small'),
            labelbottom=True
        )

        # Apply rotation using autofmt_xdate for final layout adjustment
        fig.autofmt_xdate(rotation=45, ha='right')

        # Legend Styling
        ax.legend(
            loc=params.get('legend_loc', 'upper center'),
            bbox_to_anchor=params.get('legend_bbox_to_anchor', (0.5, -0.3)),
            ncol=params.get('legend_ncol', 4),
            frameon=params.get('legend_frameon', False)
        )

        # --- Alt Text Generation ---

        # --- New Details for Accessibility ---
        # Use the *original* df for totals, not the filtered plot_df
        totals_summary_str = "No total data calculated."
        highest_total_str = ""
        try:
            # Calculate totals for each location
            totals_df = multi_timeseries_df.select(
                pl.col(location_names).sum()
            )
            totals_dict = totals_df.to_dicts()[0]

            # Handle None/null values by treating them as 0.0
            sorted_totals = sorted(
                totals_dict.items(),
                key=lambda item: (item[1] if item[1] is not None else 0.0),
                reverse=True
            )

            # Create summary strings
            summary_list = []
            for name, total in sorted_totals:
                safe_total = total if total is not None else 0.0
                summary_list.append(f"{name} ({safe_total:.2f} in)")

            totals_summary_str = "Totals: " + ", ".join(summary_list) + "."

            # Highest hourly value (Null-safe)
            max_obs_df = multi_timeseries_df.select(pl.col(location_names).max())
            max_obs_dict = max_obs_df.to_dicts()[0]

            sorted_max = sorted(
                max_obs_dict.items(),
                key=lambda item: (item[1] if item[1] is not None else 0.0),
                reverse=True
            )

            if sorted_max:
                highest_loc, highest_val = sorted_max[0]
                safe_high_val = highest_val if highest_val is not None else 0.0
                highest_total_str = (
                    f"The single highest hourly observation was "
                f"{safe_high_val:.2f} inches at {highest_loc}."
                f"**Plot may show data for multiple locations at "
                f"identical hourly intervals. Visual separation of lines "
                f"is for readability only; time values are simultaneous.**"
                )

        except Exception as e:
            print(f"Warning: Could not calculate alt text totals. Error: {e}")

        # Format date range
        start_date_str = start_dt.strftime('%Y-%m-%d') if start_dt else 'N/A'
        end_date_str = end_dt.strftime('%Y-%m-%d') if end_dt else 'N/A'
        date_range_str = f"from {start_date_str} to {end_date_str}"

        # Manual Alt Text pattern.
        alt_text_string = (
            f"Grouped stem plot titled '{title}'. "
            f"Shows data for {num_locations} locations {date_range_str}. "
            f"{totals_summary_str} {highest_total_str}"
        )
        print(f"\nAlt text: {simple_word_wrap(alt_text_string)}")

        return {'figure': fig, 'alt_text': alt_text_string}

    except Exception as e:
        print(f"❌ Error during multi-location stem plot: \n   Details: {e}")
        traceback.print_exc()
        if 'fig' in locals(): plt.close(fig) # Ensure closure on error
        return {'figure': None, 'alt_text': "Plot could not be generated."}




def plot_daily_subplots(multi_timeseries_df, start_dt, end_dt, params={}):
    """
    Generates a figure with one subplot for each day.
    *Within* each subplot, it plots a grouped stem plot for *all* locations
    and places a legend *underneath* each subplot using bbox_to_anchor
    and manual fig.subplots_adjust.

    Args:
        multi_timeseries_df (pl.DataFrame): The wide-format DataFrame.
        start_dt (datetime): The start of the full, user-defined time window.
        end_dt (datetime): The end of the full, user-defined time window.
        params (dict): A dictionary of plotting parameters.

    Returns:
        dict: A dictionary containing the figure object and alt text.
    """
    # --- Input Validation ---
    if multi_timeseries_df is None or multi_timeseries_df.is_empty():
        print("Warning: Cannot generate daily subplots because no data was provided.")
        return {'figure': None, 'alt_text': "Plot could not be generated."}

    try:
        # --- Setup: Determine Locations and Date Range ---
        location_names = params.get(
            'location_order',
            [col for col in multi_timeseries_df.columns if col != 'datetime']
        )
        num_locations = len(location_names)
        if not location_names:
            print("Warning: No location columns found in DataFrame to plot.")
            return {'figure': None, 'alt_text': "No locations to plot."}

        date_range = list(pl.date_range(
            start_dt.date(),
            end_dt.date(),
            interval="1d",
            eager=True
        ))
        num_days = len(date_range)
        if num_days == 0:
            print("Date range is less than one day. Cannot create daily subplots.")
            return {'figure': None, 'alt_text': "Date range too short."}

        # --- Get common plot params ---
        fig_width = params.get('figsize_width', 14)
        fig_height_per_plot = params.get('figsize_height_per_plot', 3.5)
        total_fig_height = fig_height_per_plot * num_days

        title_size = params.get('plot_title_size', 'large')
        title_weight = params.get('plot_title_weight', 'bold')
        ylabel = params.get('plot_ylabel', 'QPE (in)')
        ylabel_size = params.get('plot_ylabel_size', 'medium')
        grid = params.get('plot_grid', True)
        grid_alpha = params.get('plot_grid_alpha', 0.6)
        tick_spacing = params.get('y_tick_spacing', 0.1)

        # Multi-location params
        colors = params.get('colors', ['#0076D6', '#162E51',
                                       '#084fc9', '#707070'])
        markers = params.get('markers', ['o'] * num_locations)
        offset_width = params.get('stem_offset_width', 0.008)
        locations_map = params.get('locations_map', {})

        # --- Create the figure and axes array ---
        fig, axes = plt.subplots(
            nrows=num_days,
            ncols=1,
            figsize=(fig_width, total_fig_height),
            dpi=params.get('fig_dpi', None),
            squeeze=False
        )
        axes = axes.flatten()

        # --- Calculate a *global* Y-axis for all subplots ---
        all_values = multi_timeseries_df.select(
            pl.col(location_names)).to_numpy().flatten()
        valid_values = all_values[~np.isnan(all_values)]
        global_y_max = 0
        if len(valid_values) > 0:
            max_val = valid_values.max()
            if max_val > 0:
                global_y_max = (max_val // tick_spacing + 1) * tick_spacing
            else:
                global_y_max = tick_spacing
        else:
            global_y_max = 1.0

        print(f"\n--- Generating Daily Subplot Figure for {num_locations} locations "
              f"({num_days} days) ---")

        # Stats collection for Alt Text
        daily_stats = []

        # --- Loop Through Each Day (Subplot) ---
        for i in range(num_days):
            ax = axes[i]
            current_day = date_range[i]

            day_start_dt = datetime.combine(current_day, datetime.min.time())
            day_end_dt = datetime.combine(current_day, datetime.max.time())

            day_df = multi_timeseries_df.filter(
                pl.col("datetime").dt.date() == current_day
            )

            # Find max value for this day across all locs
            day_max_val = 0.0
            if not day_df.is_empty():
                # Flatten all location columns to find max
                day_vals = day_df.select(pl.col(location_names)).to_numpy().flatten()
                day_vals = day_vals[~np.isnan(day_vals)]
                if len(day_vals) > 0:
                    day_max_val = day_vals.max()

            daily_stats.append(
                f"{current_day.strftime('%Y-%m-%d')} (Max: {day_max_val:.2f} in)")

            ax.set_title(
                f"Daily Plot: {current_day.strftime('%Y-%m-%d')}",
                fontsize=title_size,
                fontweight=title_weight
            )

            if (day_df.is_empty() or
                day_df.select(pl.col(location_names)).null_count().row(0) ==
                (day_df.height * num_locations,)):
                ax.text(0.5, 0.5, 'No data for this day',
                        ha='center', va='center', transform=ax.transAxes,
                        fontstyle='italic', color='gray')
            else:
                date_nums = mdates.date2num(day_df["datetime"].to_list())

                for j, loc_name in enumerate(location_names):
                    if loc_name in day_df.columns:

                        coords = locations_map.get(loc_name, {})
                        lat = coords.get('latitude')
                        lon = coords.get('longitude')
                        if isinstance(lat, (int, float)) and isinstance(lon, (int, float)):
                            full_label = f"{loc_name} (Lat: {lat:.2f}, Lon: {lon:.2f})"
                        else:
                            full_label = f"{loc_name} (Coordinates N/A)"

                        offset = offset_width * (j - (num_locations - 1)) / 2
                        markerline, stemlines, baseline = ax.stem(
                            date_nums + offset,
                            day_df[loc_name],
                            label=full_label, # Pass the full label
                            basefmt=" "
                        )
                        current_marker = markers[j % len(markers)]
                        plt.setp(stemlines, 'color', colors[j % len(colors)], 'zorder', 3)
                        plt.setp(markerline, marker=current_marker,
                                 color=colors[j % len(colors)], zorder=3)

            # --- Apply Y-Axis Styling ---
            ax.set_ylabel(ylabel, fontsize=ylabel_size)
            ax.set_ylim(bottom=0, top=global_y_max)
            ax.grid(grid, alpha=grid_alpha, linestyle='--', axis='y')

            # --- Apply X-Axis Styling ---
            ax.set_xlim(day_start_dt, day_end_dt)
            ax.set_xlabel("Time (UTC)", fontsize=ylabel_size)

            # --- X-Axis Tick Logic ---
            (final_major_locator, final_major_formatter,
             final_minor_locator, final_minor_formatter) = _determine_xaxis_ticks(
                day_start_dt, day_end_dt, params
            )
            if final_major_locator: ax.xaxis.set_major_locator(final_major_locator)
            if final_major_formatter: ax.xaxis.set_major_formatter(final_major_formatter)
            if final_minor_locator: ax.xaxis.set_minor_locator(final_minor_locator)
            if final_minor_formatter: ax.xaxis.set_minor_formatter(final_minor_formatter)

            ax.tick_params(axis='x', which='major', **{
                'direction': params.get('x_tick_direction', 'out'),
                'length': params.get('x_major_s_tick_length', 6),
                'width': params.get('x_major_tick_width', 1.0),
                'labelsize': params.get('x_major_tick_labelsize', 'medium'),
                'labelbottom': True
            })
            ax.tick_params(axis='x', which='minor', **{
                'direction': params.get('x_tick_direction', 'out'),
                'length': params.get('x_minor_tick_length', 3),
                'width': params.get('x_minor_tick_width', 0.6),
                'labelsize': params.get('x_minor_tick_labelsize', 'x-small'),
                'labelbottom': True
            })

            plt.setp(
                ax.get_xticklabels(),
                rotation=45,
                ha='right',
                rotation_mode='anchor'
            )

            handles, labels = ax.get_legend_handles_labels()
            if handles: # Only add legend if there's data
                ax.legend(
                    handles,
                    labels,
                    loc=params.get('ax_legend_loc', 'upper center'),
                    bbox_to_anchor=params.get('ax_legend_bbox_to_anchor', (0.5, -0.35)),
                    ncol=params.get('ax_legend_ncol', 4),
                    frameon=params.get('ax_legend_frameon', False),
                    fontsize=params.get('ax_legend_fontsize', 'small')
                )

        # --- Finalize Figure (Title, Layout) ---
        fig.suptitle(
            f"Daily Time Series for {num_locations} Locations\n"
            f"From: {start_dt.strftime('%Y-%m-%d')} to {end_dt.strftime('%Y-%m-%d')}",
            fontsize=params.get('fig_title_size', 'x-large'),
            fontweight=params.get('fig_title_weight', 'bold')
        )

        # --- Dynamic Layout Adjustment Logic ---
        # Calculate dynamic 'top' margin to handle 1-day vs multi-day layouts
        # 1 Day: top=0.7 (more space), 5 Days: top=0.9 (standard space)

        # Define range
        min_days = 1
        max_days = 5  # Cap dynamic scaling at 5 days
        min_top = 0.70
        max_top = 0.90

        if num_days <= min_days:
             calculated_top = min_top
        elif num_days >= max_days:
             calculated_top = max_top
        else:
             # Linear interpolation for 2, 3, 4 days
             slope = (max_top - min_top) / (max_days - min_days)
             calculated_top = min_top + slope * (num_days - min_days)

        # Allow user override ONLY if it falls within sensible bounds (optional safety)

        # We check if user passed a specific 'subplot_top'.
        # If so, we might use it, but the we should enforce the dynamic range logic.
        # We will use the calculated value as the new default.
        final_subplot_top = calculated_top

        # Optional: Print debug info to confirm logic
        # print(f"DEBUG: Dynamic Layout - Days: {num_days}, "
        #   f"Calculated Top: {final_subplot_top:.3f}")

        fig.subplots_adjust(
            left=params.get('subplot_left', 0.08),
            right=params.get('subplot_right', 0.97),
            hspace=params.get('subplot_hspace', 0.8),
            top=final_subplot_top,  # <--- Applied Dynamic Value
            bottom=params.get('subplot_bottom', 0.1)
        )
        # ---------------------------------------------

        # --- Alt Text Generation ---
        # Manual Alt Text pattern.

        # --- New Details for Accessibility ---
        locations_str = ", ".join(location_names)
        start_date_str = start_dt.strftime('%Y-%m-%d') if start_dt else 'N/A'
        end_date_str = end_dt.strftime('%Y-%m-%d') if end_dt else 'N/A'
        date_range_str = f"from {start_date_str} to {end_date_str}"

        # Build stats string (condense if too many days)
        if len(daily_stats) > 7:
             stats_str = f"Daily maximum values ranged from {
                min([float(s.split('Max: ')[1].split(' ')[0]) for s in daily_stats])} to {
                max([float(s.split('Max: ')[1].split(' ')[0]) for s in daily_stats])} inches."
        else:
             stats_str = "Daily Maximums: " + "; ".join(daily_stats) + "."

        alt_text_string = (
            f"A figure containing {num_days} vertically stacked subplots {date_range_str}. "
            f"Locations: {locations_str}. Shared Y-axis max: {global_y_max:.2f} inches. "
            f"{stats_str} "
            f"**Plot may show data for multiple locations at identical hourly intervals. "
            f"Visual separation of lines is for readability only; time values are simultaneous.**"
        )
        print(f"\nAlt text: {simple_word_wrap(alt_text_string)}")

        return {'figure': fig, 'alt_text': alt_text_string}

    except Exception as e:
        print(f"❌ Error during daily subplot generation: \n   Details: {e}")
        traceback.print_exc()
        if 'fig' in locals():
            plt.close(fig)
        return {'figure': None, 'alt_text': "Plot could not be generated."}

def plot_site_subplots(multi_timeseries_df, start_dt, end_dt, params={}):
    """
    Generates a single figure with multiple subplots, one for each site,
    sharing a consistent time axis and a global y-axis.

    Refactored to implement all plotting logic and x-axis configuration
    directly, following the 'plot_multi_location_stem_plot' pattern.

    Args:
        multi_timeseries_df (pl.DataFrame): The wide-format DataFrame.
        start_dt (datetime): The start of the full, user-defined time window.
        end_dt (datetime): The end of the full, user-defined time window.
        params (dict): A dictionary of plotting parameters.

    Returns:
        dict: A dictionary containing the figure object and alt text.
    """
    # --- Input Validation ---
    if multi_timeseries_df is None or multi_timeseries_df.is_empty():
        print("Warning: Cannot generate site subplots because no data was provided.")
        return {'figure': None, 'alt_text': "Plot could not be generated."}

    try:
        # --- Determine Layout, Setup Figure, and Global Y-Axis ---
        location_names = params.get(
            'location_order',
            [col for col in multi_timeseries_df.columns if col != 'datetime']
        )
        num_locations = len(location_names)
        if num_locations == 0:
            print("Warning: No location columns found in DataFrame to plot.")
            return {'figure': None, 'alt_text': "No locations to plot."}

        # CALCULATE GLOBAL Y-AXIS LIMIT ---
        all_values = multi_timeseries_df.select(
            pl.col(location_names)).to_numpy().flatten()
        valid_values = all_values[~np.isnan(all_values)]

        global_y_max = 0
        tick_spacing = params.get('y_tick_spacing', 0.1)

        if len(valid_values) > 0:
            # Find the actual max value
            max_val = valid_values.max()
            if max_val > 0:
                # Calculate the ceiling limit based on tick spacing
                global_y_max = (max_val // tick_spacing + 1) * tick_spacing
            else:
                global_y_max = tick_spacing # Default to one tick if max is 0
        else:
             global_y_max = 1.0 # Default if no data at all

        # Setup the figure
        fig, axes = plt.subplots(
                    num_locations, 1,
                    figsize=(params.get('figsize_width', 12),
                             params.get('figsize_height_per_plot', 4) * num_locations),
                    dpi=params.get('fig_dpi', None),
                    squeeze=False
                )
        axes = axes.flatten()

        print(f"\n--- Generating Site Subplot Figure ({num_locations} sites) ---")

        # Get common plot params
        bar_color = params.get('color', '#0076D6')
        bar_width = params.get('bar_width', 0.03)
        title_size = params.get('plot_title_size', 'large')
        title_weight = params.get('plot_title_weight', 'bold')
        ylabel = params.get('plot_ylabel', 'QPE (in)')
        ylabel_size = params.get('plot_ylabel_size', 'medium')
        grid = params.get('plot_grid', True)
        grid_alpha = params.get('plot_grid_alpha', 0.6)

        # Stats collection
        site_stats = []

        for i, loc_name in enumerate(location_names):
            current_ax = axes[i]

            # Set title for this subplot
            current_ax.set_title(
                f"QPE for {loc_name}",
                fontsize=title_size,
                fontweight=title_weight
            )

            # --- Data Plotting ---
            if loc_name not in multi_timeseries_df.columns:
                print(f"⚠️ Warning: Skipping '{loc_name}' as it is not in the DataFrame.")
                current_ax.text(0.5, 0.5, f"No data column for {loc_name}",
                                ha='center', va='center', transform=current_ax.transAxes)
                current_ax.set_xticks([]); current_ax.set_yticks([])
                continue

            # Filter to non-null data for this specific location
            loc_df = multi_timeseries_df.select(['datetime', loc_name]).filter(
                pl.col(loc_name).is_not_null()
            )

            # Total and Max for this site (Null-safe)
            site_total = 0.0
            site_max = 0.0
            if not loc_df.is_empty():
                # Plot the data as bars
                current_ax.bar(
                    loc_df['datetime'],
                    loc_df[loc_name],
                    width=bar_width,
                    color=bar_color
                )

                val_sum = loc_df[loc_name].sum()
                site_total = val_sum if val_sum is not None else 0.0

                val_max = loc_df[loc_name].max()
                site_max = val_max if val_max is not None else 0.0

            site_stats.append(f"{loc_name} (Total: {site_total:.2f}, Max: {site_max:.2f})")

            if loc_df.is_empty():
                 current_ax.text(0.5, 0.5, "No data for this time range",
                                 ha='center', va='center', transform=current_ax.transAxes)

            # --- Apply Y-Axis Styling ---
            current_ax.set_ylabel(ylabel, fontsize=ylabel_size)
            current_ax.set_ylim(bottom=0, top=global_y_max) # Apply global Y max
            current_ax.grid(grid, alpha=grid_alpha, linestyle='--', axis='y')

            # --- Apply X-Axis Styling (Global Range) ---
            current_ax.set_xlim(start_dt, end_dt)

            # Only set the x-label on the VERY LAST subplot
            if i == num_locations - 1:
                current_ax.set_xlabel("Date", fontsize=ylabel_size) # Reuse size
            else:
                current_ax.set_xlabel("")

            # --- BEGIN X-AXIS TICK LOGIC  ---
            # Determine final locators/formatters using the helper
            final_major_locator, final_major_formatter, \
            final_minor_locator, final_minor_formatter = _determine_xaxis_ticks(
                start_dt, end_dt, params)

            # Apply the determined objects to this subplot's axis
            if final_major_locator:
                current_ax.xaxis.set_major_locator(final_major_locator)
            if final_major_formatter:
                current_ax.xaxis.set_major_formatter(final_major_formatter)
            if final_minor_locator:
                current_ax.xaxis.set_minor_locator(final_minor_locator)
            if final_minor_formatter:
                current_ax.xaxis.set_minor_formatter(final_minor_formatter)

            # Apply Tick Sizing/Styling from Params to this subplot's axis
            current_ax.tick_params(
                axis='x', which='major', direction=params.get('x_tick_direction', 'out'),
                length=params.get('x_major_tick_length', 6),
                width=params.get('x_major_tick_width', 1.0),
                labelsize=params.get('x_major_tick_labelsize', 'medium')
            )
            current_ax.tick_params(
                axis='x', which='minor', direction=params.get('x_tick_direction', 'out'),
                length=params.get('x_minor_tick_length', 3),
                width=params.get('x_minor_tick_width', 0.6),
                labelsize=params.get('x_minor_tick_labelsize', 'x-small'),
                labelbottom=True
            )

        # --- Finalize Figure ---
        fig.autofmt_xdate(rotation=45, ha='right') # Apply rotation to all subplots
        fig.tight_layout(rect=[0, 0.03, 1, 0.97]) # Adjust for titles/labels

        # --- Alt Text Generation ---
        # Manual Alt Text pattern.

        # --- Details for Accessibility ---
        locations_str = ", ".join(location_names)
        start_date_str = start_dt.strftime('%Y-%m-%d') if start_dt else 'N/A'
        end_date_str = end_dt.strftime('%Y-%m-%d') if end_dt else 'N/A'
        date_range_str = f"from {start_date_str} to {end_date_str}"

        # Add site stats to alt text
        stats_str = "Site Stats: " + "; ".join(site_stats) + "."

        alt_text_string = (
            f"A figure containing {num_locations} vertically stacked "
            f"subplots {date_range_str}. "
            f"Shared Y-axis max: {global_y_max:.2f} inches. "
            f"{stats_str}"
        )
        print(f"\nAlt text: {simple_word_wrap(alt_text_string)}")

        return {'figure': fig, 'alt_text': alt_text_string}

    except Exception as e:
        print(f"❌ Error during site subplot generation: \n   Details: {e}")
        traceback.print_exc()
        if 'fig' in locals():
            plt.close(fig)
        return {'figure': None, 'alt_text': "Plot could not be generated."}

### **2.6 Data Export Helpers**
**The Goal:**
To handle saving files. These functions generate unique, timestamped filenames (so you never overwrite old work) and trigger the browser download if you are using Google Colab.

**How to use this cell:**
* **Run the cell.** (No user input required).


In [ ]:
# --- 2.6: Data Export Helpers ---
# (Contains all logic for naming, saving, and downloading data files & plots)

def generate_descriptive_filename(prefix, params):
    """
    Generates a safe, descriptive, and UNIQUE filename from a prefix
    and parameter dict, appending a timestamp.

    Prints a legend explaining the filename characters (p, m, range, timestamp)
    to the user for clarity.

    Args:
        prefix (str): The base name for the file (e.g., "single_loc").
        params (dict): A dictionary of validated workflow parameters.

    Returns:
        str: A sanitized, descriptive, and unique base filename (no extension).
    """
    parts = [prefix]

    # Sanitize and add location
    if 'lat' in params and 'lon' in params:
        # Replace . with p (point) and - with m (minus) for filesystem safety
        lat_str = str(params.get('lat', 'N/A')).replace('.', 'p').replace('-', 'm')
        lon_str = str(params.get('lon', 'N/A')).replace('.', 'p').replace('-', 'm')
        parts.append(f"lat_{lat_str}_lon_{lon_str}")

    # Sanitize and add location list (for multi-loc)
    if 'locations' in params:
        num_locs = len(params.get('locations', []))
        parts.append(f"{num_locs}_locations")

    # Sanitize and add dates with "Start" and "End" prefixes
    start_date_str = ""
    if 'start_dt' in params:
        start_date_str = params['start_dt'].strftime('%Y%m%d')
        parts.append(f"Start_{start_date_str}")

    if 'end_dt' in params:
        end_date_str = params['end_dt'].strftime('%Y%m%d')
        # Only append End date if it differs from Start date
        if start_date_str != end_date_str:
            parts.append(f"to_End_{end_date_str}")

    # Add a precise timestamp to ensure uniqueness
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    parts.append(timestamp)

    # Join all parts with an underscore
    final_name = "_".join(parts)

    # Final sanitization: remove any other weird characters
    sanitized_name = re.sub(r'[^\w\-]', '_', final_name)

    # PRINT LEGEND: Now explicitly shows the Timestamp FORMAT
    print(f"\n[Filename Key: 'p'=decimal, 'm'=minus, Range='Start_YYYYMMDD_to_End_YYYYMMDD', Suffix='_YYYYMMDD_HHMMSS']")

    return sanitized_name


def trigger_colab_download(file_path):
    """
    Triggers a file download prompt *only* if in Google Colab.
    Otherwise, prints the local path.
    """
    # This function checks the global IS_COLAB flag set in cell 2.7
    if 'IS_COLAB' in globals() and IS_COLAB:
        try:
            from google.colab import files
            print(f"Triggering browser download for: {file_path}")
            files.download(file_path)
        except Exception as e:
            print(f"❌ Colab download failed. You can download the file manually from the")
            print(f"   file browser. Path: {file_path}")
            print(f"   Error: {e}")
    else:
        print(f"✅ File saved to your local directory:")
        print(f"   {os.path.abspath(file_path)}")


def save_data(dataframe, prefix, params, file_format='csv', is_pivoted=False, export_mode='single_file'):
    """
    Saves a DataFrame locally using a descriptive filename.
    Supports saving as a single file OR zipping individual files for multi-location data.

    Args:
        dataframe (pl.DataFrame): The data to save.
        prefix (str): The base prefix for the filename (e.g., "single_loc").
        params (dict): Dict of validated params for filename generation.
        file_format (str): The format ('csv' or 'parquet').
        is_pivoted (bool): Flag to indicate if the data is wide-format (multi-location).
        export_mode (str): 'single_file' (default) or 'individual_files' (zips separate per-column files).
    """
    if dataframe is None or dataframe.is_empty():
        print("Warning: No data to save.")
        return

    try:
        # Generate the descriptive, timestamped filename (Prints Key)
        base_filename = generate_descriptive_filename(prefix, params)
        file_format = file_format.lower()

        # --- BRANCH A: Split and Zip Individual Files (Multi-Location Only) ---
        if is_pivoted and export_mode == 'individual_files':
            zip_filename = f"{base_filename}.zip"
            print(f"Packaging data into individual {file_format} files within: '{zip_filename}'...")

            try:
                with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
                    # The first column is 'datetime'; all others are locations
                    location_cols = [col for col in dataframe.columns if col != 'datetime']

                    if not location_cols:
                        print("Warning: No location columns found to split. Saving as single file instead.")
                        pass

                    for loc_name in location_cols:
                        # Extract data for just this location (and datetime)
                        # Filter out nulls to keep the file clean if specific sites have gaps
                        site_df = dataframe.select(['datetime', loc_name])

                        # Create a safe filename for this site inside the zip
                        safe_loc_name = re.sub(r'[^\w\-]', '_', loc_name)
                        member_filename = f"{prefix}_{safe_loc_name}.{file_format}"

                        # We temporarily write to disk to add to zip (Polars/ZipFile easiest path)
                        if file_format == 'csv':
                            site_df.write_csv(member_filename)
                        else:
                            site_df.write_parquet(member_filename)

                        # Add to zip
                        zipf.write(member_filename)

                        # Clean up temp file
                        if os.path.exists(member_filename):
                            os.remove(member_filename)

                trigger_colab_download(zip_filename)
                return # Exit success

            except Exception as e:
                print(f"❌ Error creating zip file: {e}. Falling back to single file save.")
                # Fall through to standard save below

        # --- BRANCH B: Standard Single File Save ---
        if file_format == 'parquet':
            full_filename = f"{base_filename}.parquet"
            print(f"Saving data to '{full_filename}'...")
            dataframe.write_parquet(full_filename)
            trigger_colab_download(full_filename)

        elif file_format == 'csv':
            full_filename = f"{base_filename}.csv"
            print(f"Saving data to '{full_filename}'...")
            dataframe.write_csv(full_filename)
            trigger_colab_download(full_filename)

        else:
            raise ValueError(f"Unsupported file format '{file_format}'. Use 'csv' or 'parquet'.")

    except Exception as e:
        print(f"❌ Error: Failed to save data. \n   Details: {e}")
        traceback.print_exc()


def save_plots(plot_objects, prefix, params, file_format='png', dpi=150):
    """
    Saves a dictionary of Matplotlib figures locally using descriptive
    filenames, then triggers Colab downloads. Also closes figures
    to prevent memory leaks.
    """
    if not plot_objects:
        print("Warning: No plot objects to save.")
        return

    print(f"\n--- Saving {len(plot_objects)} Individual Plots ---")

    # Generate a *base* filename (Prints Key)
    base_filename = generate_descriptive_filename(prefix, params)

    saved_files = []

    for plot_name, plot_info in plot_objects.items():
        fig = plot_info.get('figure')
        if not fig or not isinstance(fig, plt.Figure):
            print(f"⚠️ Skipping '{plot_name}': No valid figure object found.")
            continue

        try:
            # Create a unique name for *this* plot
            safe_plot_name = re.sub(r'[^\w\-]', '_', plot_name)
            full_filename = f"{base_filename}_{safe_plot_name}.{file_format.lower()}"

            print(f"Saving plot: '{full_filename}'...")
            fig.savefig(full_filename, dpi=dpi, bbox_inches='tight')

            # Close the figure to free memory
            plt.close(fig)

            saved_files.append(full_filename)

        except Exception as e:
            print(f"❌ Error saving plot for '{plot_name}': {e}")
            if fig:
                plt.close(fig)

    # Trigger downloads *after* all plots are saved and closed
    if saved_files:
        print(f"\nTriggering downloads for {len(saved_files)} saved plots...")
        for f in saved_files:
            trigger_colab_download(f)

    print("--- Plot saving complete. ---")

### **2.7 Environment Helper**
**The Goal:**
To automatically detect if you are running in the cloud (Google Colab) or locally. This sets a flag that tells the notebook whether to try and auto-download files to your browser.

**How to use this cell:**
* **Run the cell.** (No user input required).

In [ ]:
# --- 2.7: Environment Helper ---

# This function performs a conditional import to safely check if we are in
# a Google Colab environment. This prevents ImportError on a local machine.
def check_colab_environment():
    """
    Checks if the notebook is running in Google Colab.

    Returns:
        bool: True if in Colab, False otherwise.
    """
    try:
        # Attempt to import the Colab-specific module
        import google.colab
        print("✅ Google Colab environment detected. Auto-download will be enabled.")
        return True
    except ImportError:
        # Module doesn't exist, so we are not in Colab
        print("✅ Local environment detected. Files will be saved to the notebook directory.")
        return False

# Set the global flag that all other functions will check
IS_COLAB = check_colab_environment()

---
## **3.0 S3 Discovery: Locating Your Data**
NOAA stores this precipitation data in an Amazon S3 Bucket. Because the dataset is massive, it is organized into a structure of "partitions" (folders).

**The Goal:**
We need to construct a valid file path to a specific year of data.

**How to use this section:**
1.  **Define Dataset:** In the cell below, ensure `DATASET_TO_EXPLORE` matches the product you want (e.g., `MultiSensor_QPE...`).
    Explore the [AWS S3 bucket](https://noaa-mrms-parquet-pds.s3.us-east-1.amazonaws.com/index.html) to find your desired product: 
    * *Default Behavior:* The path starts at the bucket root. You can access any dataset here (e.g., `FLASH_QPE...`).
    * *Trace Values:* Datasets under the `v2` folder include trace (<0.1") amounts of precipitation. Those directly under the root folder have trace values removed.
    * *Optimization Tip:* A specialized `compacted/` folder exists for some datasets. Using this folder (e.g., `compacted/MultiSensor_QPE...` or `v2/compacted/MultiSensor_QPE...`) provides significantly faster data retrieval.
    * *Warning:* Not all datasets are available in `compacted/`. If your dataset is missing there, remove the prefix to search the root.

2.  **Iterative Search:** We will use the `partition_choices` list to "drill down" into the folders.
    * Start with `partition_choices = []`.
    * Run the cell. It will list the available folders (e.g., `state=Alabama`, `state=Washington`).
    * Copy a folder name, add it to the list (e.g., `['state=Washington']`), and run it again.
    * Repeat until you find the `year=...` folder and the output says **"VALIDATION SUCCESS"**.

    

### **3.1 Step A: User Input for S3 Discovery**
**The Goal:**
To configure the search parameters for the S3 bucket.

**How to use this cell:**
1.  **`DATASET_TO_EXPLORE`**: Leave this as the default to explore the "MultiSensor_QPE" product, or change it if you know a specific dataset path.
2.  **`partition_choices`**: This is your navigation tool.
    * **To start:** Set this to `[]` (empty list) to find the sub folders that are available.
    * **To drill down:** As you discover folders in Step B, copy and paste them into this list (e.g., `['compacted', 'state=Virginia']`).

In [ ]:
# --- 3.1: User Input ---

# ==============================================================================
# 1. DATASET SELECTION
# ==============================================================================
# Define the dataset path relative to the bucket root (s3://noaa-mrms-parquet-pds/)
#
# PATH TIPS:
# - OPTIMIZED: "compacted/MultiSensor_QPE_01H_Pass2_00.00"
#   (Structure: compacted -> Dataset -> state=... -> year=...)
#
# - STANDARD:  "FLASH_QPE_FFGMAX_00.00/parquet/"
#   (Structure: Dataset -> parquet -> state=... -> year=...)

DATASET_TO_EXPLORE = "v2/compacted/MultiSensor_QPE_01H_Pass2_00.00"


# ==============================================================================
# 2. PARTITION DISCOVERY
# ==============================================================================
# These choices "drill down" into the folders.
#
# EXAMPLES:
# - For 'compacted' data:
#   partition_choices = ['state=Washington', 'year=2022']
#
# - For 'standard' (root) data:
#   partition_choices = ['state=Washington', 'year=2022']

partition_choices = ['state=Washington', 'year=2022']

### **3.2 Step B: Run Iterative Discovery**
**The Goal:**
To execute the search and see what folders exist at your current path.

**How to use this cell:**
1.  **Run the cell.**
2.  **Read the Output:**
    * **If you see a list of partitions:** (e.g., `state=Alabama`, `state=Alaska`)...
        * Choose one.
        * Go back to **Step A** and add it to your `partition_choices` list.
        * **Re-run Step A**, then **Re-run Step B**.
    * **If you see "VALIDATION SUCCESS":** You have found the data. You are ready to proceed to Section 4.0 or 5.0.

In [ ]:
# --- 3.2: Execution ---

# These variables are set by this cell and will be used by the
# *next* workflows. We explicitly reset them to None on each run.
DYNAMICALLY_VALIDATED_PATH = None
DYNAMICALLY_SELECTED_YEAR = None

try:
    # Initialize S3 connection
    s3 = get_s3_filesystem(anon=True)

    if s3 is None:
        raise ConnectionError("Failed to initialize S3. Cannot proceed.")

    # Define the base path from user config (Root-Relative Logic)
    clean_dataset_path = DATASET_TO_EXPLORE.strip('/') # Clean leading/trailing slashes
    base_s3_path = f"s3://noaa-mrms-parquet-pds/{clean_dataset_path}"

    (
        validated_path,
        selected_year,
    ) = discover_s3_path_iteratively(
        base_s3_path,
        partition_choices,  # Pass in the user config
        s3
    )

    # Set variables for the next notebook cells
    if validated_path:
        DYNAMICALLY_VALIDATED_PATH = validated_path
        DYNAMICALLY_SELECTED_YEAR = selected_year

except Exception as e:
    print(f"❌ An unexpected error occurred during the workflow: {e}")
    traceback.print_exc()

---
## **4.0 Workflow: Single Location Time Series**
This workflow extracts a precipitation time series for a **single specific point** (Latitude/Longitude) over a specific time window.

**The Goal:**
To extract, visualize, and save precipitation data for one specific geographic coordinate.

**Overview of Workflow:**
1.  **Configuration:** You define the location, time range, and output preferences in **Step A**.
2.  **Fetch:** The notebook downloads the full dataset for the discovered year from S3.
3.  **Analysis:** It calculates the distance from your target Lat/Lon to the nearest available grid point in the data (Nearest Neighbor).
4.  **Extraction:** It filters the data to just that point and time range.
5.  **Visualization:** It plots a bar chart and saves the results using the `SINGLE_LOC_FILENAME_PREFIX`.

### **4.1 Step A: User Input**
**The Goal:**
To define the specific coordinate (Lat/Lon) and time window you want to analyze.

**How to use this cell:**
1.  **`ENABLE_SAVE`**: Set to `True` if you want to download the resulting data and plot to your computer.
2.  **Location:** Update `ADV_TARGET_LATITUDE` and `ADV_TARGET_LONGITUDE` with your decimal coordinates.
3.  **Date Range:** Update the Start and End times.
    * *Note:* The **Year** is locked to the dataset you discovered in Section 3. You cannot change the year here; you must go back to Section 3 to change datasets.
4.  **Output Filenames:** You can customize the name of your saved files by changing the `SINGLE_LOC_FILENAME_PREFIX` variable. This text will form the *beginning* of the filename, followed by the coordinates and date range automatically.

In [ ]:
# --- 4.1: Configuration ---

# ==============================================================================
# 1. SINGLE LOCATION OUTPUT CONFIGURATION
# ==============================================================================
# Set ENABLE_SAVE to True to download the data and plot.
# These settings apply ONLY to the Single Location workflow.
ENABLE_SAVE = True

# Define the "Base Name" for your output files.
# The system will automatically append the coordinates, date range, and a timestamp
# to this prefix to ensure the filename is unique and descriptive.
# Example: "my_analysis" -> "my_analysis_lat_47p5_lon_m123_Start_2020... .csv"
SINGLE_LOC_FILENAME_PREFIX = "single_loc"

SINGLE_LOC_EXPORT_FORMAT = "csv"      # 'csv' or 'parquet'


# ==============================================================================
#  YEAR SELECTION (Automatic)
# ==============================================================================
# We use the year discovered in Section 3.2 to prevent mismatches.
if 'DYNAMICALLY_SELECTED_YEAR' in globals() and DYNAMICALLY_SELECTED_YEAR is not None:
    target_year = DYNAMICALLY_SELECTED_YEAR
    print(f"ℹ️ Using year from Section 3: {target_year}")
    print("   (Note: To analyze a different year, return to Section 3 and re-run discovery.)")
else:
    target_year = 2020
    print(f"⚠️ Warning: Dynamic year not set. Defaulting to: {target_year}")


# ==============================================================================
#  TARGET LOCATION
# ==============================================================================
# Specify the coordinates for your single point of interest
ADV_TARGET_LATITUDE = 47.584028
ADV_TARGET_LONGITUDE = -123.014861


# ==============================================================================
#  DATE RANGE (UTC)
# ==============================================================================
# Define the start and end of your analysis window.
# Ensure these dates fall within the 'target_year' displayed above.
ADV_START_YEAR = target_year
ADV_START_MONTH = 12
ADV_START_DAY = 21
ADV_START_HOUR = 0
ADV_START_MINUTE = 0

ADV_END_YEAR = target_year
ADV_END_MONTH = 12
ADV_END_DAY = 25
ADV_END_HOUR = 23
ADV_END_MINUTE = 59


# ==============================================================================
#  ANALYSIS PARAMETERS
# ==============================================================================
# Maximum distance (km) to accept for a "nearest neighbor" grid match.
# MRMS data is on a 0.01 x 0.01 degree (~ 1 km x 1 km) grid
ADV_DISTANCE_WARNING_THRESHOLD = 1.0


# ==============================================================================
#  PLOTTING CONFIGURATION
# ==============================================================================
adv_single_plot_params = {
    'figsize': (12, 6),
    'color': '#084fc9',
    'bar_width': 0.03,
    'grid': True,
    'y_tick_spacing': 0.1, #Use 0.1 for QPE and 10 for QPE/FFG
    'ylabel': 'QPE (in)', #Use QPE (in) or QPE/FFG (%)

    # --- X-Axis Tick Control ---
    'x_major_tick_interval': 'default',
    'x_minor_tick_interval': 'default',
    'x_tick_label_format': 'default',
    'x_minor_tick_label_format': 'default',

    # --- Tick Size/Style ---
    'x_major_tick_labelsize': 'medium',
    'x_minor_tick_labelsize': 'x-small',
    'x_major_tick_length': 6,
    'x_major_tick_width': 1.0,
    'x_minor_tick_length': 3,
    'x_minor_tick_width': 0.6,
    'x_tick_direction': 'out'
}

### **4.2 Step B: Validate Inputs**
**The Goal:**
To perform a safety check before downloading data.

**How to use this cell:**
1.  **Run the cell.**
2.  **Check the Output:**
    * It will confirm if your dates and coordinates are valid.
    * It will warn you if there is a mismatch (e.g., trying to analyze 2023 dates when you selected 2020 data).
    * **If errors occur:** Fix the variables in **Step A** and re-run that cell, then re-run this validation cell.

In [ ]:
# --- 4.2: Validation ---

# Clear state from any previous runs
valid_lat, valid_lon, valid_start_dt, valid_end_dt = (None, None, None, None)

try:
    print("--- Configuration Validation Check ---")

    # Validate the inherited state from section 3.2
    if (
        'DYNAMICALLY_SELECTED_YEAR' not in locals() or DYNAMICALLY_SELECTED_YEAR is None or
        'DYNAMICALLY_VALIDATED_PATH' not in locals() or DYNAMICALLY_VALIDATED_PATH is None
    ):
        raise ValueError(
            "Required variables (DYNAMICALLY_SELECTED_YEAR, DYNAMICALLY_VALIDATED_PATH) "
            "are not set. Please run section 3.2 successfully first."
        )

    print(f"✅ Inherited path: {DYNAMICALLY_VALIDATED_PATH}")

    # Perform the "Year Mismatch" check
    print(f"✅ Info: Validating against discovered data year: {DYNAMICALLY_SELECTED_YEAR}")
    if (ADV_START_YEAR != DYNAMICALLY_SELECTED_YEAR or
        ADV_END_YEAR != DYNAMICALLY_SELECTED_YEAR):

        print("="*80)
        print("⚠️ CONFIGURATION WARNING: YEAR MISMATCH")
        print(f"   The S3 data path you discovered in Chapter 3 is for the year: {DYNAMICALLY_SELECTED_YEAR}")
        print(f"   But your time window in this cell is set from {ADV_START_YEAR} to {ADV_END_YEAR}.")
        print("   This will likely result in an EMPTY plot.")
        print("👉 Please align the ADV_START_YEAR and ADV_END_YEAR with your discovered data.")
        print("="*80)
    else:
        print("   Year settings match discovered data. Configuration looks good.")

    # Confirm Distance Threshold is Set
    print(f"\n✅ Distance warning threshold is set to: {ADV_DISTANCE_WARNING_THRESHOLD} km.")
    print("   (The logic in Step C will use this to flag distant locations.)")
    print("----------------------------------------")

    # Validate geographic coordinates
    valid_lat, valid_lon = validate_lat_lon(ADV_TARGET_LATITUDE, ADV_TARGET_LONGITUDE)
    print(f"✅ Validated Coordinates: (Lat: {valid_lat}, Lon: {valid_lon})")

    # Construct and validate the datetime objects
    print("Validating date range...")
    start_dt = construct_and_validate_datetime(
        ADV_START_YEAR, ADV_START_MONTH, ADV_START_DAY,
        hour=ADV_START_HOUR, minute=ADV_START_MINUTE
    )
    end_dt = construct_and_validate_datetime(
        ADV_END_YEAR, ADV_END_MONTH, ADV_END_DAY,
        hour=ADV_END_HOUR, minute=ADV_END_MINUTE
    )

    if end_dt < start_dt:
        raise ValueError(
            f"Start date ({start_dt}) must be before or on "
            f"the end date ({end_dt})."
        )

    # Success: Assign to the "valid" variables for the next step
    valid_start_dt = start_dt
    valid_end_dt = end_dt
    print(f"✅ Validated Date Range: {valid_start_dt} to {valid_end_dt}")

    print("\n--- Input Validation Successful ---")

except ValueError as e:
    print(f"❌ Input Validation Failed:\n   Details: {e}")
except Exception as e:
    print(f"❌ An unexpected error occurred during validation.")
    traceback.print_exc()

### **4.3 Step C: Fetch & Analyze Data**
**The Goal:**
To perform the heavy lifting: downloading the full dataset from the cloud and extracting your specific time series.

**How to use this cell:**
1.  **Run the cell.** You do not need to make any changes for it to run.
2.  **Wait:** This may take a moment as it downloads large files.
3.  **Review the Feedback:**
    * Look for the **"Nearest Grid Cell Selection"** output.
    * Check the **Distance (km)**. If the distance is large (e.g., > 25km), the data might not accurately represent your target location.

In [ ]:
# --- 4.3: Fetch & Analyze ---


# Clear state from any previous runs
single_loc_df = None
feedback = None

try:
    if 'valid_start_dt' not in locals() or valid_start_dt is None:
        raise RuntimeError("Validation step (B) did not complete successfully. Re-run Section 4.2.")

    # Initialize filesystem
    s3 = get_s3_filesystem(anon=True)

    # Execute the high-performance pushdown analysis
    single_loc_df, feedback = run_single_location_analysis(
        s3_path=DYNAMICALLY_VALIDATED_PATH,
        s3_filesystem=s3,
        lat=valid_lat,
        lon=valid_lon,
        start_dt=valid_start_dt,
        end_dt=valid_end_dt,
        distance_threshold=ADV_DISTANCE_WARNING_THRESHOLD
    )

    if single_loc_df is not None:
        print(f"✅ Successfully extracted {len(single_loc_df)} time series records without eagerly loading the state.")
        null_count = single_loc_df.select(pl.col('observation').is_null().sum())[0, 0]
        print(f"   (Including {null_count} records with null observations - i.e., missing hours).")

except RuntimeError as e:
    print(f"❌ Error: {e}")
except Exception as e:
    print(f"❌ An unexpected error occurred.")
    traceback.print_exc()


### **4.4 Step D: Plot & Save Data**
**The Goal:**
To visualize the results and save the files.

**How to use this cell:**
1.  **Run the cell.**  You do not need to make any changes for it to run.
2.  **View the Plot:** A bar chart will appear below the cell.
3.  **Get your Files:**
    * If `ENABLE_SAVE` was `True`, check your downloads folder (or the file browser on the left if using Colab) for the CSV/Parquet data and the PNG image.

> **Tip: Customizing your Filenames**
> The filename for your saved files is defined in **Section 4.1** (Variable: `SINGLE_LOC_FILENAME_PREFIX`).
> * **Current Prefix:** defined in 4.1 (default is "single_loc")
> * **To change it:** Return to **Section 4.1**, update the variable (e.g., to `"Seattle_Analysis"`), run that cell to apply the settings, and then run this cell again.

In [ ]:
# --- 4.4: Plot & Save ---

fig = None
try:
    # Check if the analysis step (C) completed successfully
    if 'single_loc_df' not in locals() or single_loc_df is None or 'feedback' not in locals():
        raise RuntimeError(
            "Analysis step (C) did not complete successfully or found no data. "
            "Please re-run the cell above."
        )

    # Plot the data (generate the figure)
    print("Generating plot...")

    plot_params = adv_single_plot_params.copy()
    plot_params['start_datetime'] = valid_start_dt
    plot_params['end_datetime'] = valid_end_dt
    plot_params['lat'] = valid_lat
    plot_params['lon'] = valid_lon
    plot_params['title'] = (
        f"QPE for Target (Lat: {valid_lat}, Lon: {valid_lon})\n"
        f"Data from Grid Point (Lat: {feedback['selected_lat']:.4f}, Lon: {feedback['selected_lon']:.4f})"
    )

    plot_result = plot_single_location_timeseries(single_loc_df, params=plot_params)

    fig = plot_result.get('figure')
    if fig is None:
        raise RuntimeError("Plotting function failed to return a figure object.")

    # Show the plot for immediate feedback
    plt.show()

    # Save and download plot and data if enabled
    if ENABLE_SAVE:
        # Build params dict for descriptive filename
        file_params = {
            'lat': valid_lat,
            'lon': valid_lon,
            'start_dt': valid_start_dt,
            'end_dt': valid_end_dt
        }

        # --- Save the PLOT ---
        plot_prefix = f"{SINGLE_LOC_FILENAME_PREFIX}_plot"
        plot_filename = generate_descriptive_filename(plot_prefix, file_params) + ".png"

        print(f"Saving plot to '{plot_filename}'...")
        fig.savefig(plot_filename, dpi=150, bbox_inches='tight')
        trigger_colab_download(plot_filename) # Trigger download

        # --- Save the DATA ---
        if not single_loc_df.is_empty():
            print("\nSaving data...")
            save_data(
                dataframe=single_loc_df,
                prefix=SINGLE_LOC_FILENAME_PREFIX,
                params=file_params,
                file_format=SINGLE_LOC_EXPORT_FORMAT,
                is_pivoted=False
            )
        else:
            print("Info: No data to save.")
    else:
        print("\nInfo: `ENABLE_SAVE` is set to False. Skipping save.")

except RuntimeError as e:
    print(f"❌ Error: {e}")
except Exception as e:
    print(f"❌ An unexpected error occurred during plotting or saving.")
    traceback.print_exc()
finally:
    # Always close the figure to prevent memory leaks
    if fig:
        plt.close(fig)

---
## **5.0 Advanced Workflow: Multi-Location Time Series**
This workflow scales up the previous analysis. Instead of one point, you can provide a dictionary of **multiple locations**.

**The Goal:**
To compare precipitation events across multiple locations simultaneously.

**Overview of Workflow:**
1.  **Configuration:** You define a list of named sites (e.g., "Site A", "Site B") and their coordinates in **Step A**.
2.  **Batch Processing:** The notebook finds the nearest grid point for *every* site in your list.
3.  **Data Structuring:** It pivots the data into a "Wide Format" table (one column per site, one row per hour).
4.  **Visualization:** It generates three specialized comparison plots:
    * **Grouped Stem Plot:** A combined view of all locations on one timeline.
    * **Daily Subplots:** A breakdown of all locations day-by-day.
    * **Site-by-Site Subplots:** Individual charts for each location stacked vertically.

### **5.1 Step A: User Input Configuration**
**The Goal:**
To define a batch of locations for comparative analysis.

**How to use this cell:**
1.  **`ADV_MULTI_LOCATION_TARGETS`**: Edit this dictionary to include your specific sites.
    * Format: `"Name": {"latitude": 12.34, "longitude": -56.78}`
2.  **`EXPORT_MODE`**:
    * Set to `"single_file"` to get one wide CSV with columns for every site.
    * Set to `"individual_files"` to get a ZIP file containing separate CSVs for each site.
3.  **Date Range:** Set the Start and End times for the analysis window.
4.  **Output Filenames:** You can customize the name of your saved files by changing the `MULTI_LOC_FILENAME_PREFIX` variable. This text will form the *beginning* of the filename.

In [ ]:
# --- 5.1: Configuration ---

# ==============================================================================
# 1. MULTI-LOCATION OUTPUT CONFIGURATION
# ==============================================================================
# These settings apply ONLY to the Multi-Location workflow.
ENABLE_SAVE = True

# Define the "Base Name" for your output files.
# The system will automatically append details (like locations and dates) and a timestamp
# to this prefix to ensure the filename is unique.
# Example: "region_study" -> "region_study_5_locations_Start_2020... .csv"
MULTI_LOC_FILENAME_PREFIX = "multi_loc"

MULTI_LOC_EXPORT_FORMAT = "csv"        # 'csv' or 'parquet'

# Export Mode:
# 'single_file' = One CSV/Parquet with columns for each location.
# 'individual_files' = A ZIP file containing separate files for each location.
EXPORT_MODE = "individual_files"


# ==============================================================================
# 2. YEAR SELECTION (Automatic)
# ==============================================================================
if 'DYNAMICALLY_SELECTED_YEAR' in globals() and DYNAMICALLY_SELECTED_YEAR is not None:
    target_year = DYNAMICALLY_SELECTED_YEAR
    print(f"ℹ️ Using year from Section 3: {target_year}")
    print("   (Note: To analyze a different year, return to Section 3 and re-run discovery.)")
else:
    target_year = 2020
    print(f"⚠️ Warning: Dynamic year not set. Defaulting to: {target_year}")


# ==============================================================================
# 3. TIME PERIOD (UTC)
# ==============================================================================
ADV_MULTI_START_YEAR = target_year
ADV_MULTI_START_MONTH = 12
ADV_MULTI_START_DAY = 21
ADV_MULTI_START_HOUR = 0
ADV_MULTI_START_MINUTE = 0

ADV_MULTI_END_YEAR = target_year
ADV_MULTI_END_MONTH = 12
ADV_MULTI_END_DAY = 26
ADV_MULTI_END_HOUR = 23
ADV_MULTI_END_MINUTE = 59


# ==============================================================================
# 4. LOCATION PARAMETERS
# ==============================================================================
# Define your sites here. Keys (e.g., "Site_A") will become column headers.
ADV_MULTI_LOCATION_TARGETS =  {
    "Site_A": {"latitude": 47.950500, "longitude": -124.350750},
    "Site_B": {"latitude": 47.431722, "longitude": -123.853639},
    "Site_C": {"latitude": 47.631722, "longitude": -123.899339},
    "Site_D": {"latitude": 47.584028, "longitude": -123.014861},
    "Site_E": {"latitude": 46.584028, "longitude": -121.014861}
}


# ==============================================================================
# 5. ANALYSIS & PLOTTING PARAMETERS
# ==============================================================================
# MRMS data is on a 0.01 x 0.01 degree (~ 1 km x 1 km) grid
ADV_MULTI_DISTANCE_WARNING_THRESHOLD = 25.0 # (in kilometers)

default_location_order = list(ADV_MULTI_LOCATION_TARGETS.keys())

ADV_MULTI_COMMON_PLOT_PARAMS = {
    'location_order': default_location_order,
    'colors': ['#0076D6', '#162E51', '#707070', '#d95f02', '#6ba539'],
    'markers': ['o', 's', '^', 'd', 'v', '*', 'X']
}

### **5.2 Step B: Validate Inputs**
**The Goal:**
To ensure all locations in your list are formatted correctly and the dates are valid.

**How to use this cell:**
1.  **Run the cell.**
2.  **Troubleshooting:** If the validation fails, check that every item in your `ADV_MULTI_LOCATION_TARGETS` dictionary has both a `latitude` and `longitude` key.

In [ ]:
# --- 5.2: Validation ---

# Clear state from any previous runs
adv_valid_start_dt, adv_valid_end_dt = (None, None)
adv_valid_locations = None

try:
    print("--- Configuration Validation Check ---")

    # Validate the inherited state from section 2.5.1
    if (
        'DYNAMICALLY_SELECTED_YEAR' not in locals() or DYNAMICALLY_SELECTED_YEAR is None or
        'DYNAMICALLY_VALIDATED_PATH' not in locals() or DYNAMICALLY_VALIDATED_PATH is None
    ):
        raise ValueError(
            "Required variables (DYNAMICALLY_SELECTED_YEAR, DYNAMICALLY_VALIDATED_PATH) "
            "are not set. Please run section 3.2 successfully first."
        )

    print(f"✅ Inherited path: {DYNAMICALLY_VALIDATED_PATH}")

    # Perform the "Year Mismatch" check
    print(f"✅ Info: Validating against discovered data year: {DYNAMICALLY_SELECTED_YEAR}")
    if (ADV_MULTI_START_YEAR != DYNAMICALLY_SELECTED_YEAR or
        ADV_MULTI_END_YEAR != DYNAMICALLY_SELECTED_YEAR):

        print("="*80)
        print("⚠️ CONFIGURATION WARNING: YEAR MISMATCH")
        print(f"   The S3 data path you discovered in Chapter 2.5 is for the year: {DYNAMICALLY_SELECTED_YEAR}")
        print(f"   But your time window in this cell is set from {ADV_MULTI_START_YEAR} to {ADV_MULTI_END_YEAR}.")
        print("   This will likely result in an EMPTY plot.")
        print("👉 Please align the ADV_MULTI_START_YEAR and ADV_MULTI_END_YEAR with your discovered data.")
        print("="*80)
    else:
        print("   Year settings match discovered data. Configuration looks good.")

    print("----------------------------------------")

    # Validate the locations dictionary structure
    print("Validating locations dictionary...")
    if (
        not isinstance(ADV_MULTI_LOCATION_TARGETS, dict) or
        not ADV_MULTI_LOCATION_TARGETS
    ):
        raise ValueError(
            "'ADV_MULTI_LOCATION_TARGETS' must be a non-empty dictionary."
        )

    first_key = next(iter(ADV_MULTI_LOCATION_TARGETS))
    first_loc = ADV_MULTI_LOCATION_TARGETS[first_key]
    if (
        not isinstance(first_loc, dict) or
        'latitude' not in first_loc or
        'longitude' not in first_loc
    ):
        raise ValueError(
            "Each item in 'ADV_MULTI_LOCATION_TARGETS' must be a dictionary "
            "containing 'latitude' and 'longitude' keys."
        )

    print(f"✅ Validated {len(ADV_MULTI_LOCATION_TARGETS)} target locations.")
    adv_valid_locations = ADV_MULTI_LOCATION_TARGETS

    # Construct and validate the datetime objects
    # Uses the high-granularity hour/minute variables
    print("\nValidating date range...")
    start_dt = construct_and_validate_datetime(
        ADV_MULTI_START_YEAR, ADV_MULTI_START_MONTH, ADV_MULTI_START_DAY,
        hour=ADV_MULTI_START_HOUR, minute=ADV_MULTI_START_MINUTE
    )
    end_dt = construct_and_validate_datetime(
        ADV_MULTI_END_YEAR, ADV_MULTI_END_MONTH, ADV_MULTI_END_DAY,
        hour=ADV_MULTI_END_HOUR, minute=ADV_MULTI_END_MINUTE
    )

    if end_dt < start_dt:
        raise ValueError(
            f"Start date ({start_dt}) must be before or on "
            f"the end date ({end_dt})."
        )

    # Success: Assign to the "valid" variables for the next step
    adv_valid_start_dt = start_dt
    adv_valid_end_dt = end_dt
    print(f"✅ Validated Date Range: {adv_valid_start_dt} to {adv_valid_end_dt}")

    print("\n--- Input Validation Successful ---")

except ValueError as e:
    print(f"❌ Input Validation Failed:\n   Details: {e}")
except Exception as e:
    print(f"❌ An unexpected error occurred during validation.")
    traceback.print_exc()

### **5.3 Step C: Fetch, Analyze, & Save Data**
**The Goal:**
To download the data once and process it for *all* locations simultaneously.

**How to use this cell:**
1.  **Run the cell.** You do not need to make any changes for it to run.
2.  **Review the Table:** The output will show a table listing every location, its nearest grid point, and the distance.
    * *Tip:* Scan the "Distance" column. If any site shows a warning, consider checking its coordinates.
3.  **Export:** If save is enabled, your CSV/Parquet files will be generated here.

> **Tip: Customizing your Filenames**
> The filename for your saved files is defined in **Section 5.1** (Variable: `MULTI_LOC_FILENAME_PREFIX`).
> * **Current Prefix:** defined in 5.1 (default is "multi_loc")
> * **To change it:** Return to **Section 5.1**, update the variable, run that cell to apply the settings, and then run this cell again.

In [ ]:
# --- 5.3: Fetch, Analyze, & Save ---

adv_multi_timeseries_df = None
adv_multi_nearest_points_df = None

try:
    if 'adv_valid_start_dt' not in locals() or adv_valid_start_dt is None:
        raise RuntimeError("Validation step (B) did not complete successfully. Re-run Section 5.2.")

    s3 = get_s3_filesystem(anon=True)

    # Execute the high-performance multi-location analysis
    adv_multi_timeseries_df, adv_multi_nearest_points_df = run_multi_location_analysis(
        s3_path=DYNAMICALLY_VALIDATED_PATH,
        s3_filesystem=s3,
        locations_dict=adv_valid_locations,
        start_dt=adv_valid_start_dt,
        end_dt=adv_valid_end_dt,
        value_col="observation",
        distance_threshold=ADV_MULTI_DISTANCE_WARNING_THRESHOLD
    )

    if adv_multi_timeseries_df is not None and not adv_multi_timeseries_df.is_empty():
        print("\nSaving pivoted multi-location data...")
        file_params = {
            'locations': adv_valid_locations,
            'start_dt': adv_valid_start_dt,
            'end_dt': adv_valid_end_dt
        }

        save_data(
            dataframe=adv_multi_timeseries_df,
            prefix=f"{MULTI_LOC_FILENAME_PREFIX}",
            params=file_params,
            file_format=MULTI_LOC_EXPORT_FORMAT,
            is_pivoted=True,
            export_mode=EXPORT_MODE
        )
        print("\n--- Data extraction and saving complete. Ready for plotting. ---")
    else:
        print("\nInfo: No data was loaded from S3.")

except Exception as e:
    print(f"❌ An unexpected error occurred during data analysis: {e}")
    traceback.print_exc()

---
### **5.4 Multi-Location Plotting**
**The Goal:**
To generate three distinct visualizations of your multi-site data.

**How to use this section:**
The cells below (5.4.1, 5.4.2, 5.4.3) are independent. You can run one, two, or all of them depending on how you want to view the data.

**File Naming Note:**
Each plot below will be saved using the `MULTI_LOC_FILENAME_PREFIX` you defined in **Section 5.1**, with a unique suffix appended (e.g., `_stem_plot`, `_daily_subplots`, `_site_subplots`).

#### **5.4.1 Plot 1 - Grouped Stem Plot**
**The Goal:**
To visualize all locations on a **single timeline** to easily compare the timing of precipitation events.

**How to use this cell:**
1.  **Run the cell.** You do not need to make any changes for it to run. There are parameters you can change to customize your plot.
2.  **Interpret the Plot:** Each location is represented by a "stem" (vertical line). This is best for identifying if rain occurred simultaneously at all sites or if it moved across the region over time.

In [ ]:
# --- 5.4.1: Plot 1 - Grouped Stem Plot (Advanced) ---
# This cell generates, displays, and optionally saves a grouped stem plot.
fig = None
try:
    # Check if the required data is available
    if 'adv_multi_timeseries_df' not in locals() or adv_multi_timeseries_df is None or adv_multi_timeseries_df.is_empty():
        raise RuntimeError(
            "The 'adv_multi_timeseries_df' DataFrame is not available. "
            "Please successfully run '5.3 Step C' first."
        )

    #  Load the COMMON plot parameters from Step A
    # Contains 'location_order', 'colors', 'markers'
    final_plot_params = ADV_MULTI_COMMON_PLOT_PARAMS.copy()

    # Determine State Name for Title
    # We parse this from the S3 path defined in Section 3.
    # Format is typically: .../state=Washington/...
    state_name = "Selected Region" # Default fallback
    if 'DYNAMICALLY_VALIDATED_PATH' in globals() and DYNAMICALLY_VALIDATED_PATH:
        import re
        # Find the text after 'state=' but before the next slash
        match = re.search(r'state=([^/]+)', DYNAMICALLY_VALIDATED_PATH)
        if match:
            state_name = match.group(1)

    # Define and merge LOCAL, plot-specific parameters
    local_plot_params = {
        'figsize': (12, 8),
        'y_tick_spacing': 0.1, # Use 0.1 for QPE or 10 for QPE/FFG
        'locations_map': adv_valid_locations, # Uses validated locations
        'stem_offset_width': 0.008,
        'state': state_name, # (✅) Pass the parsed state name here
        'plot_ylabel' : 'QPE (in)', #Use QPE (in) or QPE/FFG (%)

        # --- X-Axis Tick Control Parameters ---
        'x_major_tick_interval': 'default', # Examples: '6H', '1D', '2W'. Use None to disable.
        'x_minor_tick_interval': 'default', # Examples: '12H', '1D'. Use None to disable.
        'x_tick_label_format': 'default', # Examples: "%Y-%m-%d %H:%M", "%Y-%m-%d". Use None to disable.
        'x_minor_tick_label_format': 'default', # Example: "%H". Use None to disable.

        # --- Tick Size/Style Parameters ---
        'x_major_tick_labelsize': 'medium',
        'x_minor_tick_labelsize': 'x-small',
        'x_major_tick_length': 6,
        'x_major_tick_width': 1.0,
        'x_minor_tick_length': 3,
        'x_minor_tick_width': 0.6,
        'x_tick_direction': 'out',

        # --- Legend Parameters ---
        'legend_loc': 'upper center',
        'legend_bbox_to_anchor': (0.5, -0.2),
        'legend_ncol': 4,
        'legend_frameon': False
    }

    # The .update() call merges the two dictionaries
    final_plot_params.update(local_plot_params)

    # Generate the plot (but don't show it yet)
    print("\n--- Generating Multi-Location Stem Plot (Advanced) ---")
    plot_result = plot_multi_location_stem_plot(
        adv_multi_timeseries_df,
        adv_valid_start_dt,      # Uses variable from the logic cell
        adv_valid_end_dt,        # Uses variable from the logic cell
        params=final_plot_params # Uses the final MERGED params
    )

    fig = plot_result.get('figure')
    if fig is None:
        raise RuntimeError("Plotting function failed to return a figure object.")

    # Show the plot for immediate feedback
    plt.show()

    # Save and download the plot if enabled
    if ENABLE_SAVE:
        # Build params dict for descriptive filename
        file_params = {
            'locations': adv_valid_locations,
            'start_dt': adv_valid_start_dt,
            'end_dt': adv_valid_end_dt
        }

        # Generate a unique name for this specific plot (updated prefix)
        plot_prefix = f"{MULTI_LOC_FILENAME_PREFIX}_stem_plot"
        filename = generate_descriptive_filename(plot_prefix, file_params) + ".png"

        print(f"Saving plot to '{filename}'...")
        fig.savefig(filename, dpi=150, bbox_inches='tight')

        # Trigger download (if in Colab)
        trigger_colab_download(filename)

except RuntimeError as e:
    print(f"❌ Prerequisite Error: {e}")
except Exception as e:
    print(f"❌ An unexpected error occurred during plot generation.")
    traceback.print_exc()
finally:
    # Always close the figure to prevent memory leaks
    if fig:
        plt.close(fig)


#### **5.4.2 Plot 2 - Multi-Location Daily Bar Plots**
**The Goal:**
To break the timeline down **day-by-day** for clearer visibility.

**How to use this cell:**
1.  **Run the cell.** You do not need to make any changes for it to run. There are parameters you can change to customize your plot.
2.  **Interpret the Plot:** This generates a separate subplot for each day in your date range. It is useful when analyzing long durations (e.g., a month) where a single chart would be too crowded.

In [ ]:
# --- 5.4.2: Plot 2 - Multi-Location Daily Bar Plots (Advanced) ---

fig = None
try:
    # Check if the required data is available
    if 'adv_multi_timeseries_df' not in locals() or adv_multi_timeseries_df is None or adv_multi_timeseries_df.is_empty():
        raise RuntimeError(
            "The 'adv_multi_timeseries_df' DataFrame is not available. "
            "Please successfully run '5.3 Step C' first."
        )

    # Load the COMMON plot parameters from Step A
    # Contains 'location_order', 'colors', 'markers'
    final_plot_params = ADV_MULTI_COMMON_PLOT_PARAMS.copy()

    # Define and merge LOCAL, plot-specific parameters
    local_plot_params = {
        # --- Figure-level ---
        'figsize_width': 14,
        'figsize_height_per_plot': 3.3,
        'locations_map': adv_valid_locations,

        # --- Subplot Data Plotting (Multi-Location) ---
        'stem_offset_width': 0.008,
        'y_tick_spacing': 0.1, #Use 0.1 for QPE or 10 for QPE/FFG
        'plot_title_size': 'large',
        'plot_title_weight': 'bold',
        'plot_ylabel': 'QPE (in)', #Use QPE (in) or QPE/FFG (%)
        'plot_ylabel_size': 'medium',
        'plot_grid': True,
        'plot_grid_alpha': 0.6,

        # --- X-Axis Tick Control Parameters ---
        'x_major_tick_interval': 'default',
        'x_minor_tick_interval': 'default',
        'x_tick_label_format': 'default',
        'x_minor_tick_label_format': 'default',

        # --- Tick Size/Style Parameters ---
        'x_major_tick_labelsize': 'medium',
        'x_minor_tick_labelsize': 'x-small',
        'x_major_tick_length': 6,
        'x_major_tick_width': 1.0,
        'x_minor_tick_length': 3,
        'x_minor_tick_width': 0.6,
        'x_tick_direction': 'out',

        # --- Axis-Level Legend beneath each daily plot---
        'ax_legend_loc': 'upper center',
        'ax_legend_bbox_to_anchor': (0.5, -0.75),
        'ax_legend_ncol': 4,
        'ax_legend_fontsize': 'small',
        'ax_legend_frameon': False,

        # ---Manual Layout ---
        'subplot_hspace': 1.3,
        'subplot_top': 0.9,
        'subplot_bottom': 0.1,
        'subplot_left': 0.07,
        'subplot_right': 0.97
    }

    # The .update() call merges the two dictionaries
    final_plot_params.update(local_plot_params)

    # --- Generate the daily subplot figure ---
    print("\n--- Generating Daily Subplot Figure (Advanced) ---")
    plot_result = plot_daily_subplots(
        adv_multi_timeseries_df,
        adv_valid_start_dt,
        adv_valid_end_dt,
        params=final_plot_params
    )

    fig = plot_result.get('figure')
    if fig is None:
        raise RuntimeError("Plotting function failed to return a figure object.")

    # Show the plot for immediate feedback
    plt.show()

    # Save and download the plot if enabled
    if ENABLE_SAVE:
        # Build params dict for descriptive filename
        file_params = {
            'locations': adv_valid_locations,
            'start_dt': adv_valid_start_dt,
            'end_dt': adv_valid_end_dt
        }

        # Generate a unique name for this specific plot
        plot_prefix = f"{MULTI_LOC_FILENAME_PREFIX}_daily_subplots"
        filename = generate_descriptive_filename(plot_prefix, file_params) + ".png"

        print(f"Saving plot to '{filename}'...")
        fig.savefig(filename, dpi=150, bbox_inches='tight')

        # Trigger download (if in Colab)
        trigger_colab_download(filename)

except RuntimeError as e:
    print(f"❌ Prerequisite Error: {e}")
except Exception as e:
    print(f"❌ An unexpected error occurred during plot generation.")
    traceback.print_exc()
finally:
    # Always close the figure to prevent memory leaks
    if fig:
        plt.close(fig)

#### **5.4.3 Plot 3 - Site-by-Site Subplots**
**The Goal:**
To isolate each location's data into its own chart.

**How to use this cell:**
1.  **Run the cell.** You do not need to make any changes for it to run. There are parameters you can change to customize your plot.
2.  **Interpret the Plot:** This creates a vertical stack of charts, one for each site. This is the best view for analyzing the intensity/magnitude of the event at specific individual sites without visual clutter from other locations.

In [ ]:
# --- 5.4.3: Plot 3 - Site-by-Site Subplots (Advanced) ---

fig = None
try:
    # Check if the required data is available
    if 'adv_multi_timeseries_df' not in locals() or adv_multi_timeseries_df is None or adv_multi_timeseries_df.is_empty():
        raise RuntimeError(
            "The 'adv_multi_timeseries_df' DataFrame is not available. "
            "Please successfully run '5.3 Step C' first."
        )

    # Load the COMMON plot parameters from Step A
    # Contains 'location_order', 'colors', 'markers'
    final_plot_params = ADV_MULTI_COMMON_PLOT_PARAMS.copy()

    # Define and merge LOCAL, plot-specific parameters
    local_plot_params = {
        # --- Figure-level ---
        'figsize_width': 12,
        'figsize_height_per_plot': 3.5,
        'locations_map': adv_valid_locations,

        # 'location_order' is inherited
        # 'state' is not used in the dynamic workflow

        # --- Subplot Data Plotting ---
        'color': '#0076D6', # Note: This plot uses a single 'color', not 'colors'
        'bar_width': 0.03,
        'y_tick_spacing': 0.1, #Use 0.1 for QPE or 10 for QPE/FFG
        'plot_title_size': 'large',
        'plot_title_weight': 'bold',
        'plot_ylabel': 'QPE (in)', #Use QPE (in) or QPE/FFG (%)
        'plot_ylabel_size': 'medium',
        'plot_grid': True,
        'plot_grid_alpha': 0.6,

        # --- X-Axis Tick Control Parameters ---
        'x_major_tick_interval': 'default',
        'x_minor_tick_interval': 'default',
        'x_tick_label_format': 'default',
        'x_minor_tick_label_format': 'default',

        # --- Tick Size/Style Parameters ---
        'x_major_tick_labelsize': 'medium',
        'x_minor_tick_labelsize': 'x-small',
        'x_major_tick_length': 6,
        'x_major_tick_width': 1.0,
        'x_minor_tick_length': 3,
        'x_minor_tick_width': 0.6,
        'x_tick_direction': 'out'
    }

    # The .update() call merges the two dictionaries
    final_plot_params.update(local_plot_params)

    # --- Generate the site-by-site subplot figure ---
    print("\n--- Generating Site Subplot Figure (Advanced) ---")
    plot_result = plot_site_subplots(
        adv_multi_timeseries_df,
        adv_valid_start_dt,
        adv_valid_end_dt,
        params=final_plot_params
    )

    fig = plot_result.get('figure')
    if fig is None:
        raise RuntimeError("Plotting function failed to return a figure object.")

    # Show the plot for immediate feedback
    plt.show()

    # Save and download the plot if enabled
    if ENABLE_SAVE:
        # Build params dict for descriptive filename
        file_params = {
            'locations': adv_valid_locations,
            'start_dt': adv_valid_start_dt,
            'end_dt': adv_valid_end_dt
        }

        # Generate a unique name for this specific plot
        plot_prefix = f"{MULTI_LOC_FILENAME_PREFIX}_site_subplots"
        filename = generate_descriptive_filename(plot_prefix, file_params) + ".png"

        print(f"Saving plot to '{filename}'...")
        fig.savefig(filename, dpi=150, bbox_inches='tight')

        # Trigger download (if in Colab)
        trigger_colab_download(filename)

except RuntimeError as e:
    print(f"❌ Prerequisite Error: {e}")
except Exception as e:
    print(f"❌ An unexpected error occurred during plot generation.")
    traceback.print_exc()
finally:
    # Always close the figure to prevent memory leaks
    if fig:
        plt.close(fig)

For more information on the U.S. Precipitation Time Series Explorer product, see the [PTE Landing Page](https://www.ncei.noaa.gov/products/precipitation-time-series-explorer/). 